In [ ]:
import pandas as pd

In [ ]:
import pandas as pd

# ── All CSV files ──────────────────────────────────────────────────────────────
df_admin       = pd.read_csv("admin.csv",               encoding='latin-1')
df_bi          = pd.read_csv("billing_insurance.csv",   encoding='latin-1')
df_da          = pd.read_csv("doctor_appointments.csv", encoding='latin-1')
df_emergency   = pd.read_csv("emergency.csv",           encoding='latin-1')
df_pharma      = pd.read_csv("pharmacy.csv",            encoding='latin-1')

df_pharma.head()

,topic,question,answer
0,prescription,How do I collect a prescription medicine from ...,Take your prescription slip to the AIIMS outpa...
1,prescription,How long is a prescription from AIIMS valid fo...,Most AIIMS prescriptions are valid for a defin...
2,prescription,Can I collect a prescription on behalf of a pa...,A family member or authorised representative m...
3,prescription,What should I do if the AIIMS pharmacy does no...,Ask the pharmacist to issue a shortage slip so...
4,prescription,Can a doctor at AIIMS prescribe medicines for ...,Doctors may issue repeat prescriptions for sta...


In [ ]:
df_admin_router = df_admin[["question"]].copy()
df_billing_router = df_bi[["question"]].copy()
df_doctor_router = df_da[["question"]].copy()
df_emergency_router = df_emergency[["question"]].copy()
df_pharma_router = df_pharma[["question"]].copy()

df_admin_router["label"] = "Admin"
df_billing_router["label"] = "Billing"
df_doctor_router["label"] = "Doctor_Appointment"
df_emergency_router["label"] = "Emergency"
df_pharma_router["label"] = "Pharmacy"


df_router = pd.concat(
    [
        df_admin_router,
        df_billing_router,
        df_doctor_router,
        df_emergency_router,
        df_pharma_router
    ],
    ignore_index=True
)

In [ ]:
df_router.head()

,question,label
0,How do I register as a patient for the first t...,Admin
1,Where is the patient registration counter loca...,Admin
2,What documents should I bring when registering...,Admin
3,Can someone register a patient who cannot come...,Admin
4,How long does patient registration normally take?,Admin


In [ ]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split


def prepare_data(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Cleans the 'question' column, deduplicates, shuffles, then splits into
    train (70%), validation (15%), and test (15%) sets with no leakage.

    Args:
        df : DataFrame with 'question' and 'label' columns

    Returns:
        train_df, val_df, test_df
    """

    def clean_question(text: str) -> str:
        if not isinstance(text, str):
            text = str(text)
        text = text.lower()
        text = re.sub(r'[\$£€](\d)', r'CURRENCY\1', text)
        text = re.sub(r'(?<=[a-z0-9])\.(?=[a-z0-9])', 'DOT', text)
        text = re.sub(r'(?<=[a-z0-9])-(?=[a-z0-9])', 'HYPHEN', text)
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = text.replace('DOT', '.').replace('HYPHEN', '-').replace('CURRENCY', '$')
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    # --- Clean ---
    df = df.copy()
    df['question'] = df['question'].apply(clean_question)

    # --- Deduplicate ----------------------------------------------------------
    before = len(df)
    df     = df.drop_duplicates(subset='question').reset_index(drop=True)
    print(f"Removed {before - len(df)} duplicates ({before} → {len(df)} rows)")

    # --- Shuffle BEFORE splitting ---------------------------------------------
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"Shuffled {len(df)} rows")

    # --- Split ----------------------------------------------------------------
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
    val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    # --- Verify zero leakage --------------------------------------------------
    train_q = set(train_df['question'])
    val_q   = set(val_df['question'])
    test_q  = set(test_df['question'])

    print(f"\nLeakage check:")
    print(f"  Train  ∩ Val  : {len(train_q & val_q)}  (should be 0)")
    print(f"  Train  ∩ Test : {len(train_q & test_q)}  (should be 0)")
    print(f"  Val    ∩ Test : {len(val_q   & test_q)}  (should be 0)")

    print(f"\nFinal split:")
    print(f"  Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")
    print(f"\nClass distribution (train):\n{train_df['label'].value_counts()}")

    return train_df, val_df, test_df




In [ ]:
train_router_df, val_router_df, test_router_df = prepare_data(df_router)

Removed 0 duplicates (1950 → 1950 rows)
Shuffled 1950 rows

Leakage check:
  Train  ∩ Val  : 0  (should be 0)
  Train  ∩ Test : 0  (should be 0)
  Val    ∩ Test : 0  (should be 0)

Final split:
  Train : 1365 | Val : 292 | Test : 293

Class distribution (train):
label
Emergency             280
Pharmacy              280
Billing               280
Doctor_Appointment    280
Admin                 245
Name: count, dtype: int64


In [ ]:
#Router
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

# ── Label Encoding ─────────────────────────────────────────────────────────────
le = LabelEncoder()
le.fit(train_router_df['label'])

train_labels = le.transform(train_router_df['label'])
val_labels   = le.transform(val_router_df['label'])
test_labels  = le.transform(test_router_df['label'])

# ── Sentence Embeddings ────────────────────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')   # fast, lightweight, strong

print("Encoding splits — this takes ~30 seconds on CPU...")
train_emb = embedder.encode(
    train_router_df['question'].tolist(),
    batch_size      = 64,
    show_progress_bar = True,
    convert_to_numpy  = True
)
val_emb = embedder.encode(
    val_router_df['question'].tolist(),
    batch_size        = 64,
    show_progress_bar = True,
    convert_to_numpy  = True
)
test_emb = embedder.encode(
    test_router_df['question'].tolist(),
    batch_size        = 64,
    show_progress_bar = True,
    convert_to_numpy  = True
)

# ── Router Model ───────────────────────────────────────────────────────────────
router = LogisticRegression(
    max_iter     = 1000,
    class_weight = 'balanced',
    C            = 1.0,
    solver       = 'lbfgs',
    multi_class  = 'multinomial',
    tol          = 1e-4
)

router.fit(train_emb, train_labels)

# ── Metrics ────────────────────────────────────────────────────────────────────
def print_metrics(split_name, embeddings, true_labels):
    preds        = router.predict(embeddings)
    true_decoded = le.inverse_transform(true_labels)
    pred_decoded = le.inverse_transform(preds)
    acc          = accuracy_score(true_labels, preds)

    print(f"\n{'='*60}")
    print(f"  Metrics — {split_name}  |  Accuracy: {acc:.4f}")
    print(f"{'='*60}")
    print(classification_report(
        true_decoded,
        pred_decoded,
        target_names = le.classes_,
        digits       = 4
    ))

# ── Overfitting Gap Check ──────────────────────────────────────────────────────
def overfitting_check():
    train_acc = accuracy_score(train_labels, router.predict(train_emb))
    val_acc   = accuracy_score(val_labels,   router.predict(val_emb))
    gap       = train_acc - val_acc
    print(f"\n{'='*60}")
    print(f"  Train Accuracy : {train_acc:.4f}")
    print(f"  Val   Accuracy : {val_acc:.4f}")
    print(f"  Overfit Gap    : {gap:.4f}  {'⚠️  Overfit' if gap > 0.05 else '✅ Healthy'}")
    print(f"{'='*60}")

print_metrics("Train",      train_emb,  train_labels)
print_metrics("Validation", val_emb,    val_labels)
print_metrics("Test",       test_emb,   test_labels)
overfitting_check()

# ── Save embedder for inference ────────────────────────────────────────────────
# Call embedder.encode([user_query]) at runtime to route new questions

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



  Metrics — Train  |  Accuracy: 0.9656
                    precision    recall  f1-score   support

             Admin     0.9031    0.9510    0.9264       245
           Billing     0.9857    0.9821    0.9839       280
Doctor_Appointment     0.9549    0.9071    0.9304       280
         Emergency     0.9964    0.9893    0.9928       280
          Pharmacy     0.9824    0.9964    0.9894       280

          accuracy                         0.9656      1365
         macro avg     0.9645    0.9652    0.9646      1365
      weighted avg     0.9661    0.9656    0.9656      1365


  Metrics — Validation  |  Accuracy: 0.9384
                    precision    recall  f1-score   support

             Admin     0.8393    0.9038    0.8704        52
           Billing     1.0000    0.9500    0.9744        60
Doctor_Appointment     0.9107    0.8500    0.8793        60
         Emergency     0.9833    0.9833    0.9833        60
          Pharmacy     0.9524    1.0000    0.9756        60

          

In [ ]:
# ── Save Router Model Files ────────────────────────────────────────────────────
import pickle
import os

# Create a folder to keep everything organised
os.makedirs('saved_models/router', exist_ok=True)

# ── Save 1: The Logistic Regression Router ────────────────────────────────────
with open('saved_models/router/hospital_router.pkl', 'wb') as f:
    pickle.dump(router, f)
print("✅ Saved: hospital_router.pkl")

# ── Save 2: The Label Encoder ─────────────────────────────────────────────────
with open('saved_models/router/hospital_label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print("✅ Saved: hospital_label_encoder.pkl")

# ── Verify They Load Correctly Before Downloading ─────────────────────────────
with open('saved_models/router/hospital_router.pkl', 'rb') as f:
    test_router_load = pickle.load(f)

with open('saved_models/router/hospital_label_encoder.pkl', 'rb') as f:
    test_le_load = pickle.load(f)

# Quick sanity check
test_query_emb = embedder.encode(["What is the registration process at AIIMS?"])
test_pred      = test_router_load.predict(test_query_emb)
test_label     = test_le_load.inverse_transform(test_pred)[0]
print(f"\n✅ Verification passed — test query classified as: {test_label}")
print(f"   Router classes: {test_le_load.classes_}")

# ── Download Both Files ───────────────────────────────────────────────────────
from google.colab import files

files.download('saved_models/router/hospital_router.pkl')
files.download('saved_models/router/hospital_label_encoder.pkl')

print("\n✅ Both files downloaded successfully.")
print("   → hospital_router.pkl")
print("   → hospital_label_encoder.pkl")


## What You Will Get



✅ Saved: hospital_router.pkl
✅ Saved: hospital_label_encoder.pkl

✅ Verification passed — test query classified as: Admin
   Router classes: ['Admin' 'Billing' 'Doctor_Appointment' 'Emergency' 'Pharmacy']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Both files downloaded successfully.
   → hospital_router.pkl
   → hospital_label_encoder.pkl


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter

def prepare_model_dataset(
    df,
    text_col     = 'question',
    answer_col   = 'answer',
    label_col    = 'topic',
    test_size    = 0.3,
    val_size     = 0.5,
    random_state = 42,
    min_samples  = 2,
    verbose      = True
):
    """
    Universal dataset preparation function for all sub-models.
    Cleans, validates, and splits any model dataframe consistently.

    Args:
        df           : Raw dataframe (df_admin, df_billing, etc.)
        text_col     : Column containing input questions
        answer_col   : Column containing target answers
        label_col    : Column containing sub-topic labels
        test_size    : Fraction for temp split (val + test)
        val_size     : Fraction of temp split for validation
        random_state : Seed for reproducibility
        min_samples  : Minimum samples per label to keep
        verbose      : Print summary stats

    Returns:
        train_df, val_df, test_df  — cleaned and split dataframes
    """

    model_name = df[label_col].iloc[0] if label_col not in df.columns else \
                 f"{len(df[label_col].unique())}-topic dataset"

    if verbose:
        print(f"\n{'='*60}")
        print(f"  Dataset Preparation")
        print(f"{'='*60}")
        print(f"  Raw shape        : {df.shape}")

    # ── Step 1 : Column Validation ─────────────────────────────────────────────
    required = [text_col, answer_col, label_col]
    missing  = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}. "
                         f"Available: {df.columns.tolist()}")

    # ── Step 2 : Copy & Reset Index ───────────────────────────────────────────
    df = df[required].copy().reset_index(drop=True)

    # ── Step 3 : Drop Nulls ───────────────────────────────────────────────────
    before = len(df)
    df     = df.dropna(subset=required)
    if verbose and len(df) < before:
        print(f"  Dropped nulls    : {before - len(df)} rows")

    # ── Step 4 : Clean Text ───────────────────────────────────────────────────
    df[text_col]   = (df[text_col]
                      .astype(str)
                      .str.strip()
                      .str.replace(r'\s+', ' ', regex=True))

    df[answer_col] = (df[answer_col]
                      .astype(str)
                      .str.strip()
                      .str.replace(r'\s+', ' ', regex=True))

    df[label_col]  = (df[label_col]
                      .astype(str)
                      .str.strip()
                      .str.lower()
                      .str.replace(' ', '_'))

    # ── Step 5 : Drop Duplicates ──────────────────────────────────────────────
    before = len(df)
    df     = df.drop_duplicates(subset=[text_col])
    if verbose and len(df) < before:
        print(f"  Dropped dupes    : {before - len(df)} rows")

    # ── Step 6 : Drop Empty Strings ──────────────────────────────────────────
    before = len(df)
    df     = df[(df[text_col] != '') & (df[answer_col] != '')]
    if verbose and len(df) < before:
        print(f"  Dropped empty    : {before - len(df)} rows")

    # ── Step 7 : Drop Labels With Too Few Samples ─────────────────────────────
    label_counts  = Counter(df[label_col])
    valid_labels  = {l for l, c in label_counts.items() if c >= min_samples}
    dropped_labels= {l for l, c in label_counts.items() if c < min_samples}

    if dropped_labels and verbose:
        print(f"  Dropped labels   : {dropped_labels} (< {min_samples} samples)")

    df = df[df[label_col].isin(valid_labels)].reset_index(drop=True)

    # ── Step 8 : Length Stats ─────────────────────────────────────────────────
    df['_q_len'] = df[text_col].str.split().str.len()
    df['_a_len'] = df[answer_col].str.split().str.len()

    if verbose:
        print(f"  Clean shape      : {df.shape}")
        print(f"  Labels           : {sorted(df[label_col].unique())}")
        print(f"\n  Samples per label:")
        for label, count in sorted(Counter(df[label_col]).items()):
            bar = '█' * (count // 5)
            print(f"    {label:25s} : {count:>4}  {bar}")
        print(f"\n  Question length  : "
              f"mean={df['_q_len'].mean():.1f} | "
              f"max={df['_q_len'].max()} | "
              f"min={df['_q_len'].min()}")
        print(f"  Answer length    : "
              f"mean={df['_a_len'].mean():.1f} | "
              f"max={df['_a_len'].max()} | "
              f"min={df['_a_len'].min()}")

    # Drop helper columns
    df = df.drop(columns=['_q_len', '_a_len'])

    # ── Step 9 : Stratified Split ─────────────────────────────────────────────
    train_df, temp_df = train_test_split(
        df,
        test_size    = test_size,
        random_state = random_state,
        stratify     = df[label_col]
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size    = val_size,
        random_state = random_state,
        stratify     = temp_df[label_col]
    )

    # Reset all indices
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    # ── Step 10 : Split Summary ───────────────────────────────────────────────
    if verbose:
        print(f"\n  Split Summary:")
        print(f"    Train : {len(train_df):>4} rows  "
              f"({len(train_df)/len(df)*100:.1f}%)")
        print(f"    Val   : {len(val_df):>4} rows  "
              f"({len(val_df)/len(df)*100:.1f}%)")
        print(f"    Test  : {len(test_df):>4} rows  "
              f"({len(test_df)/len(df)*100:.1f}%)")

        print(f"\n  Stratification Check (label % per split):")
        all_labels = sorted(df[label_col].unique())
        print(f"    {'Label':25s} | {'Train':>6} | {'Val':>6} | {'Test':>6}")
        print(f"    {'-'*50}")
        for label in all_labels:
            tr = (train_df[label_col] == label).sum() / len(train_df) * 100
            vl = (val_df[label_col]   == label).sum() / len(val_df)   * 100
            ts = (test_df[label_col]  == label).sum() / len(test_df)  * 100
            print(f"    {label:25s} | {tr:>5.1f}% | {vl:>5.1f}% | {ts:>5.1f}%")

        print(f"\n{'='*60}\n")

    return train_df, val_df, test_df

In [ ]:
train_admin, val_admin, test_admin = prepare_model_dataset(df_admin)



  Dataset Preparation
  Raw shape        : (350, 3)
  Clean shape      : (350, 5)
  Labels           : ['admission', 'complaints', 'discharge', 'navigation', 'records', 'registration', 'waiting_list']

  Samples per label:
    admission                 :   50  ██████████
    complaints                :   50  ██████████
    discharge                 :   45  █████████
    navigation                :   55  ███████████
    records                   :   50  ██████████
    registration              :   50  ██████████
    waiting_list              :   50  ██████████

  Question length  : mean=10.4 | max=20 | min=5
  Answer length    : mean=21.9 | max=30 | min=12

  Split Summary:
    Train :  245 rows  (70.0%)
    Val   :   52 rows  (14.9%)
    Test  :   53 rows  (15.1%)

  Stratification Check (label % per split):
    Label                     |  Train |    Val |   Test
    --------------------------------------------------
    admission                 |  14.3% |  13.5% |  15.1%
    compla

In [ ]:
pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=1aaec2ecea64fa1ca10f65596092632ad76d1c3a4c1dc191e652b89d315d7dca
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
#Model - admin - biobart
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    BartForConditionalGeneration, BartTokenizer
)
from rouge_score import rouge_scorer
import numpy as np
import math
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Device ─────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Splits (already in memory) ────────────────────────────────────────────────
# train_admin, val_admin, test_admin

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1 — FROZEN ENCODER + RETRIEVAL INDEX
# ══════════════════════════════════════════════════════════════════════════════

ENC_MODEL     = 'sentence-transformers/all-MiniLM-L6-v2'
enc_tokenizer = AutoTokenizer.from_pretrained(ENC_MODEL)
enc_model     = AutoModel.from_pretrained(ENC_MODEL).to(device)

for param in enc_model.parameters():
    param.requires_grad = False

print(f"Encoder        : {ENC_MODEL}")
print(f"Encoder params : "
      f"{sum(p.numel() for p in enc_model.parameters()):,}  (frozen)")

# ── Frozen Encoder Module ──────────────────────────────────────────────────────
class FrozenEncoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def mean_pool(self, output, attention_mask):
        tokens   = output.last_hidden_state
        mask_exp = attention_mask.unsqueeze(-1).expand(tokens.size()).float()
        return torch.sum(tokens * mask_exp, 1) / \
               torch.clamp(mask_exp.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            output = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )
        return self.mean_pool(output, attention_mask)

encoder = FrozenEncoder(enc_model).to(device)

# ── Encode Helper ──────────────────────────────────────────────────────────────
def encode_texts(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i : i + batch_size]
        encoded = enc_tokenizer(
            batch,
            max_length     = 128,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        embs = encoder(
            encoded['input_ids'].to(device),
            encoded['attention_mask'].to(device)
        )
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

# ── Build Retrieval Index from all splits ─────────────────────────────────────
print("\nBuilding retrieval index...")
full_df       = pd.concat(
    [train_admin, val_admin, test_admin],
    ignore_index=True
).reset_index(drop=True)

full_qa_texts = (full_df['question'] + ' ' + full_df['answer']).tolist()
full_qa_embs  = encode_texts(full_qa_texts)

print(f"Indexed {len(full_df)} Q&A pairs across "
      f"{full_df['topic'].nunique()} topics")
for topic, count in full_df['topic'].value_counts().items():
    print(f"  {topic:25s} : {count} rows")

# ── Retrieval Function ─────────────────────────────────────────────────────────
def retrieve_top_k(query: str, k: int = 3) -> list:
    query_emb = encode_texts([query])
    q_norm    = nn.functional.normalize(
                    torch.tensor(query_emb,    dtype=torch.float32), dim=-1)
    k_norm    = nn.functional.normalize(
                    torch.tensor(full_qa_embs, dtype=torch.float32), dim=-1)
    sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
    top_k_idx = torch.topk(sims, k=k).indices.tolist()

    return [{
        'question'   : full_df.iloc[i]['question'],
        'answer'     : full_df.iloc[i]['answer'],
        'topic'      : full_df.iloc[i]['topic'],
        'similarity' : round(sims[i].item(), 4)
    } for i in top_k_idx]

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2 — BIOBART COMPOSER
# ══════════════════════════════════════════════════════════════════════════════

BART_MODEL     = 'GanjinZero/biobart-v2-base'
bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL)
bart_model     = BartForConditionalGeneration.from_pretrained(
                     BART_MODEL).to(device)

print(f"\nComposer       : {BART_MODEL}")
print(f"Composer params: "
      f"{sum(p.numel() for p in bart_model.parameters()):,}")

# ── Dataset ────────────────────────────────────────────────────────────────────
class AdminRAGDataset(Dataset):
    def __init__(self, df, retrieve_fn, k=3):
        self.data    = df.reset_index(drop=True)
        self.samples = self._build_samples(retrieve_fn, k)

    def _build_samples(self, retrieve_fn, k):
        samples = []
        for idx in range(len(self.data)):
            row         = self.data.loc[idx]
            context     = retrieve_fn(row['question'], k=k)
            context_str = ' '.join([
                f"Context {i+1}: {c['answer']}"
                for i, c in enumerate(context)
            ])
            samples.append({
                'input'    : (f"Question: {row['question']} "
                              f"{context_str} "
                              f"Compose a helpful hospital admin answer:"),
                'target'   : row['answer'],
                'topic'    : row['topic'],
                'question' : row['question']
            })
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# ── Collator ───────────────────────────────────────────────────────────────────
class RAGCollator:
    def __init__(self, tokenizer, max_input_len=1024, max_target_len=256):
        self.tokenizer      = tokenizer
        self.max_input_len  = max_input_len
        self.max_target_len = max_target_len

    def __call__(self, batch):
        inputs    = [b['input']    for b in batch]
        targets   = [b['target']   for b in batch]
        topics    = [b['topic']    for b in batch]
        questions = [b['question'] for b in batch]

        inp_enc = self.tokenizer(
            inputs,
            max_length     = self.max_input_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        tgt_enc = self.tokenizer(
            targets,
            max_length     = self.max_target_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )

        labels = tgt_enc['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids'      : inp_enc['input_ids'],
            'attention_mask' : inp_enc['attention_mask'],
            'labels'         : labels,
            'topics'         : topics,
            'questions'      : questions,
            'raw_targets'    : targets
        }

collator = RAGCollator(bart_tokenizer)

# ── DataLoaders ────────────────────────────────────────────────────────────────
print("\nBuilding RAG datasets...")
train_dataset = AdminRAGDataset(train_admin, retrieve_top_k, k=3)
val_dataset   = AdminRAGDataset(val_admin,   retrieve_top_k, k=3)
test_dataset  = AdminRAGDataset(test_admin,  retrieve_top_k, k=3)

train_loader  = DataLoader(train_dataset, batch_size=4,
                            shuffle=True,  collate_fn=collator)
val_loader    = DataLoader(val_dataset,   batch_size=4,
                            shuffle=False, collate_fn=collator)
test_loader   = DataLoader(test_dataset,  batch_size=4,
                            shuffle=False, collate_fn=collator)

print(f"Train : {len(train_dataset)} samples | {len(train_loader)} batches")
print(f"Val   : {len(val_dataset)}   samples | {len(val_loader)}   batches")
print(f"Test  : {len(test_dataset)}  samples | {len(test_loader)}  batches")

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 3 — TRAINING ENGINE
# ══════════════════════════════════════════════════════════════════════════════

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer    = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.min_lr       = min_lr
        self.current_step = 0
        self.base_lrs     = [pg['lr'] for pg in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        for pg, lr in zip(self.optimizer.param_groups, self._compute_lr()):
            pg['lr'] = lr

    def _compute_lr(self):
        s = self.current_step
        lrs = []
        for base_lr in self.base_lrs:
            if s < self.warmup_steps:
                lr = base_lr * (s / max(1, self.warmup_steps))
            else:
                progress = (s - self.warmup_steps) / \
                           max(1, self.total_steps - self.warmup_steps)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * \
                     (1 + math.cos(math.pi * progress))
            lrs.append(lr)
        return lrs

class AdminRAGTrainer:
    def __init__(self, model, optimizer, scheduler, tokenizer,
                 device, accum_steps=4, max_grad_norm=1.0,
                 checkpoint_path='best_admin_rag.pt'):
        self.model           = model
        self.optimizer       = optimizer
        self.scheduler       = scheduler
        self.tokenizer       = tokenizer
        self.device          = device
        self.accum_steps     = accum_steps
        self.max_grad_norm   = max_grad_norm
        self.checkpoint_path = checkpoint_path
        self.best_val_loss   = float('inf')
        self.patience_count  = 0

    def train_epoch(self, loader):
        self.model.train()
        total_loss, total_tokens = 0.0, 0
        self.optimizer.zero_grad()

        for step, batch in enumerate(loader):
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels         = batch['labels'].to(self.device)

            outputs        = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels
            )
            loss           = outputs.loss / self.accum_steps
            loss.backward()

            n_tokens       = (labels != -100).sum().item()
            total_loss    += outputs.loss.item() * n_tokens
            total_tokens  += n_tokens

            if (step + 1) % self.accum_steps == 0 or \
               (step + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.max_grad_norm
                )
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()

        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity

    def evaluate(self, loader):
        self.model.eval()
        total_loss, total_tokens  = 0.0, 0
        all_preds, all_refs       = [], []
        all_topics, all_questions = [], []

        with torch.no_grad():
            for batch in loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['labels'].to(self.device)

                outputs        = self.model(
                    input_ids      = input_ids,
                    attention_mask = attention_mask,
                    labels         = labels
                )
                n_tokens       = (labels != -100).sum().item()
                total_loss    += outputs.loss.item() * n_tokens
                total_tokens  += n_tokens

                generated = self.model.generate(
                    input_ids            = input_ids,
                    attention_mask       = attention_mask,
                    max_new_tokens       = 256,
                    num_beams            = 4,
                    early_stopping       = True,
                    no_repeat_ngram_size = 3,
                    length_penalty       = 1.0
                )

                preds     = self.tokenizer.batch_decode(
                    generated, skip_special_tokens=True
                )
                label_ids = labels.clone()
                label_ids[label_ids == -100] = self.tokenizer.pad_token_id
                refs      = self.tokenizer.batch_decode(
                    label_ids, skip_special_tokens=True
                )

                all_preds.extend(preds)
                all_refs.extend(refs)
                all_topics.extend(batch['topics'])
                all_questions.extend(batch['questions'])

        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity, all_preds, all_refs, \
               all_topics, all_questions

    def check_early_stop(self, val_loss, patience=3):
        if val_loss < self.best_val_loss:
            self.best_val_loss  = val_loss
            self.patience_count = 0
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : val_loss
            }, self.checkpoint_path)
            return False
        self.patience_count += 1
        return self.patience_count >= patience

    def load_best(self):
        if not os.path.exists(self.checkpoint_path):
            print("⚠️  No checkpoint found — saving current state...")
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : self.best_val_loss
            }, self.checkpoint_path)
        ckpt = torch.load(self.checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt['model_state'])
        print(f"✅ Best checkpoint restored — "
              f"Val Loss: {ckpt['val_loss']:.4f}")

# ── Metrics ────────────────────────────────────────────────────────────────────
rouge_scorer_obj = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
)

def compute_rouge(preds, refs):
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = rouge_scorer_obj.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    return {
        'rouge1' : round(np.mean(r1), 4),
        'rouge2' : round(np.mean(r2), 4),
        'rougeL' : round(np.mean(rL), 4)
    }

def compute_token_f1(pred, ref):
    pred_tokens = pred.lower().split()
    ref_tokens  = ref.lower().split()
    common      = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    p = len(common) / len(pred_tokens) if pred_tokens else 0
    r = len(common) / len(ref_tokens)  if ref_tokens  else 0
    return round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0.0

def print_metrics(split_name, loss, ppl, preds, refs, topics, questions):
    rouge     = compute_rouge(preds, refs)
    token_f1s = [compute_token_f1(p, r) for p, r in zip(preds, refs)]
    em        = sum(p.strip().lower() == r.strip().lower()
                    for p, r in zip(preds, refs)) / len(preds)

    print(f"\n{'='*60}")
    print(f"  Metrics — {split_name}")
    print(f"{'='*60}")
    print(f"  Loss        : {loss:.4f}")
    print(f"  Perplexity  : {ppl:.4f}")
    print(f"  ROUGE-1     : {rouge['rouge1']:.4f}")
    print(f"  ROUGE-2     : {rouge['rouge2']:.4f}")
    print(f"  ROUGE-L     : {rouge['rougeL']:.4f}")
    print(f"  Token F1    : {np.mean(token_f1s):.4f}")
    print(f"  Exact Match : {em:.4f}")
    print(f"\n  Per-Topic ROUGE-L:")
    for topic in sorted(set(topics)):
        mask    = [t == topic for t in topics]
        t_preds = [p for p, m in zip(preds, mask) if m]
        t_refs  = [r for r, m in zip(refs,  mask) if m]
        t_rouge = compute_rouge(t_preds, t_refs)
        t_f1    = np.mean([compute_token_f1(p, r)
                           for p, r in zip(t_preds, t_refs)])
        print(f"    {topic:20s} : ROUGE-L {t_rouge['rougeL']:.4f} | "
              f"Token F1 {t_f1:.4f} | n={len(t_preds)}")

# ── Initialise ─────────────────────────────────────────────────────────────────
EPOCHS       = 15
ACCUM_STEPS  = 4
total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = total_steps // 10

optimizer = torch.optim.AdamW(
    bart_model.parameters(), lr=3e-5, weight_decay=0.01
)
scheduler = WarmupCosineScheduler(optimizer, warmup_steps, total_steps)
trainer   = AdminRAGTrainer(
    bart_model, optimizer, scheduler,
    bart_tokenizer, device,
    accum_steps     = ACCUM_STEPS,
    checkpoint_path = 'best_admin_rag.pt'
)

# ── Training Loop ──────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  Training Admin BioBART RAG  |  {EPOCHS} epochs  |  "
      f"Warmup {warmup_steps}  |  Total {total_steps} steps")
print(f"{'='*70}")
print(f"  {'Epoch':>5} | {'Train Loss':>10} | {'Train PPL':>9} | "
      f"{'Val Loss':>8} | {'Val PPL':>7} | {'ROUGE-L':>7} | Status")
print(f"  {'-'*70}")

for epoch in range(EPOCHS):
    tr_loss, tr_ppl       = trainer.train_epoch(train_loader)
    vl_loss, vl_ppl, \
    vl_preds, vl_refs, \
    vl_topics, vl_qs      = trainer.evaluate(val_loader)
    vl_rouge              = compute_rouge(vl_preds, vl_refs)
    stop                  = trainer.check_early_stop(vl_loss, patience=3)
    status                = "← best ✅" if trainer.patience_count == 0 \
                            else f"patience {trainer.patience_count}/3"

    print(f"  {epoch+1:>5} | {tr_loss:>10.4f} | {tr_ppl:>9.4f} | "
          f"{vl_loss:>8.4f} | {vl_ppl:>7.4f} | "
          f"{vl_rouge['rougeL']:>7.4f} | {status}")

    if stop:
        print(f"\n  Early stopping at epoch {epoch+1}")
        break

# ══════════════════════════════════════════════════════════════════════════════
#  METRICS — ALL SPLITS + SAMPLE PREDICTIONS
# ══════════════════════════════════════════════════════════════════════════════

trainer.load_best()

tr_loss, tr_ppl, tr_preds, \
tr_refs, tr_topics, tr_qs  = trainer.evaluate(train_loader)

vl_loss, vl_ppl, vl_preds, \
vl_refs, vl_topics, vl_qs  = trainer.evaluate(val_loader)

ts_loss, ts_ppl, ts_preds, \
ts_refs, ts_topics, ts_qs  = trainer.evaluate(test_loader)

print_metrics("Train",      tr_loss, tr_ppl,
              tr_preds, tr_refs, tr_topics, tr_qs)
print_metrics("Validation", vl_loss, vl_ppl,
              vl_preds, vl_refs, vl_topics, vl_qs)
print_metrics("Test",       ts_loss, ts_ppl,
              ts_preds, ts_refs, ts_topics, ts_qs)

# ── Sample Predictions from Test Set ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Sample Predictions — Test Set (5 random)")
print(f"{'='*60}")

for i in np.random.choice(len(ts_preds), 5, replace=False):
    r = compute_rouge([ts_preds[i]], [ts_refs[i]])
    f = compute_token_f1(ts_preds[i], ts_refs[i])
    print(f"\n  Topic     : {ts_topics[i]}")
    print(f"  Question  : {ts_qs[i]}")
    print(f"  Reference : {ts_refs[i][:150]}...")
    print(f"  Generated : {ts_preds[i][:150]}...")
    print(f"  ROUGE-L   : {r['rougeL']:.4f} | Token F1 : {f:.4f}")
    print(f"  {'-'*55}")

Device : cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder        : sentence-transformers/all-MiniLM-L6-v2
Encoder params : 22,713,216  (frozen)

Building retrieval index...
Indexed 350 Q&A pairs across 7 topics
  navigation                : 55 rows
  records                   : 50 rows
  admission                 : 50 rows
  complaints                : 50 rows
  registration              : 50 rows
  waiting_list              : 50 rows
  discharge                 : 45 rows


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/666M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


Composer       : GanjinZero/biobart-v2-base
Composer params: 166,404,864

Building RAG datasets...
Train : 245 samples | 62 batches
Val   : 52   samples | 13   batches
Test  : 53  samples | 14  batches

  Training Admin BioBART RAG  |  15 epochs  |  Warmup 22  |  Total 225 steps
  Epoch | Train Loss | Train PPL | Val Loss | Val PPL | ROUGE-L | Status
  ----------------------------------------------------------------------
      1 |     1.0230 |    2.7814 |   0.0961 |  1.1009 |  0.9865 | ← best ✅
      2 |     0.0730 |    1.0757 |   0.0556 |  1.0572 |  0.9925 | ← best ✅
      3 |     0.0164 |    1.0165 |   0.0175 |  1.0176 |  0.9940 | ← best ✅
      4 |     0.0045 |    1.0045 |   0.0104 |  1.0105 |  0.9929 | ← best ✅
      5 |     0.0009 |    1.0009 |   0.0083 |  1.0083 |  0.9949 | ← best ✅
      6 |     0.0006 |    1.0006 |   0.0083 |  1.0083 |  0.9953 | patience 1/3
      7 |     0.0029 |    1.0029 |   0.0086 |  1.0086 |  0.9953 | patience 2/3
      8 |     0.0003 |    1.0003 |   0.0

In [ ]:
# ── Save Admin Model Files ─────────────────────────────────────────────────────
import os
import numpy as np
from google.colab import files

os.makedirs('saved_models/admin', exist_ok=True)

# ── Save 1: Trained BioBART Weights ───────────────────────────────────────────
torch.save(
    bart_model.state_dict(),
    'saved_models/admin/admin_biobart_weights.pt'
)
print("✅ Saved: admin_biobart_weights.pt")

# ── Save 2: Retrieval Embeddings ───────────────────────────────────────────────
np.save(
    'saved_models/admin/admin_qa_embeddings.npy',
    full_qa_embs
)
print("✅ Saved: admin_qa_embeddings.npy")

# ── Save 3: Q&A Data (the actual questions and answers for retrieval) ──────────
full_df.to_csv(
    'saved_models/admin/admin_qa_data.csv',
    index=False
)
print("✅ Saved: admin_qa_data.csv")

# ── Verify Everything Before Downloading ──────────────────────────────────────
print("\n── Verification ──────────────────────────────────────────────────────")

# Check 1: weights file exists and has content
weights_size = os.path.getsize('saved_models/admin/admin_biobart_weights.pt')
print(f"   admin_biobart_weights.pt  : {weights_size / 1e6:.1f} MB")

# Check 2: embeddings shape is correct
test_emb_load = np.load('saved_models/admin/admin_qa_embeddings.npy')
print(f"   admin_qa_embeddings.npy   : shape {test_emb_load.shape}")

# Check 3: csv rows match embeddings rows
test_csv_load = pd.read_csv('saved_models/admin/admin_qa_data.csv')
print(f"   admin_qa_data.csv         : {len(test_csv_load)} rows, "
      f"columns: {list(test_csv_load.columns)}")

# Check 4: embeddings rows == csv rows (must match for retrieval to work)
assert test_emb_load.shape[0] == len(test_csv_load), \
    f"❌ MISMATCH — embeddings {test_emb_load.shape[0]} rows " \
    f"vs csv {len(test_csv_load)} rows"
print(f"\n✅ Verification passed — embeddings and csv rows match perfectly")

# ── Quick End-to-End Inference Test Before Downloading ────────────────────────
print("\n── Quick Inference Test ──────────────────────────────────────────────")

# Reload weights into a fresh model instance to simulate PyCharm loading
from transformers import BartForConditionalGeneration, BartTokenizer

test_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
test_model.load_state_dict(
    torch.load(
        'saved_models/admin/admin_biobart_weights.pt',
        map_location=device
    )
)
test_model.to(device)
test_model.eval()

# Run one test query through the full pipeline
test_query      = "How do I register as a new patient at AIIMS?"
test_query_emb  = encode_texts([test_query])

# Retrieve top 3
import torch.nn.functional as F
q_norm    = F.normalize(
    torch.tensor(test_query_emb,  dtype=torch.float32), dim=-1
)
k_norm    = F.normalize(
    torch.tensor(test_emb_load,   dtype=torch.float32), dim=-1
)
sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
top_k_idx = torch.topk(sims, k=3).indices.tolist()

context_str = ' '.join([
    f"Context {i+1}: {test_csv_load.iloc[idx]['answer']}"
    for i, idx in enumerate(top_k_idx)
])

input_text = (
    f"Question: {test_query} "
    f"{context_str} "
    f"Compose a helpful hospital admin answer:"
)

inputs = bart_tokenizer(
    input_text,
    max_length     = 1024,
    truncation     = True,
    return_tensors = 'pt'
).to(device)

with torch.no_grad():
    output = test_model.generate(
        input_ids            = inputs['input_ids'],
        attention_mask       = inputs['attention_mask'],
        max_new_tokens       = 256,
        num_beams            = 4,
        early_stopping       = True,
        no_repeat_ngram_size = 3,
        length_penalty       = 1.0
    )

test_answer = bart_tokenizer.decode(output[0], skip_special_tokens=True)
print(f"   Query  : {test_query}")
print(f"   Answer : {test_answer[:200]}...")
print(f"\n✅ Inference test passed — model loads and generates correctly")

# ── Download All Three Files ───────────────────────────────────────────────────
print("\n── Downloading Files ─────────────────────────────────────────────────")
files.download('saved_models/admin/admin_biobart_weights.pt')
files.download('saved_models/admin/admin_qa_embeddings.npy')
files.download('saved_models/admin/admin_qa_data.csv')

print("\n✅ All 3 Admin files downloaded.")
print("   → admin_biobart_weights.pt")
print("   → admin_qa_embeddings.npy")
print("   → admin_qa_data.csv")


✅ Saved: admin_biobart_weights.pt
✅ Saved: admin_qa_embeddings.npy
✅ Saved: admin_qa_data.csv

── Verification ──────────────────────────────────────────────────────
   admin_biobart_weights.pt  : 666.1 MB
   admin_qa_embeddings.npy   : shape (350, 384)
   admin_qa_data.csv         : 350 rows, columns: ['question', 'answer', 'topic']

✅ Verification passed — embeddings and csv rows match perfectly

── Quick Inference Test ──────────────────────────────────────────────


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

   Query  : How do I register as a new patient at AIIMS?
   Answer : New patients should visit the patient registration desk where staff create a patient ID and record personal details in the hospital system....

✅ Inference test passed — model loads and generates correctly

── Downloading Files ─────────────────────────────────────────────────


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All 3 Admin files downloaded.
   → admin_biobart_weights.pt
   → admin_qa_embeddings.npy
   → admin_qa_data.csv


In [ ]:
train_bi, val_bi, test_bi = prepare_model_dataset(df_bi)


  Dataset Preparation
  Raw shape        : (400, 3)
  Clean shape      : (400, 5)
  Labels           : ['billing_disputes', 'financial_aid', 'hospital_bills', 'insurance_claims', 'insurance_policy']

  Samples per label:
    billing_disputes          :   74  ██████████████
    financial_aid             :   75  ███████████████
    hospital_bills            :   85  █████████████████
    insurance_claims          :   85  █████████████████
    insurance_policy          :   81  ████████████████

  Question length  : mean=11.4 | max=19 | min=5
  Answer length    : mean=25.3 | max=38 | min=18

  Split Summary:
    Train :  280 rows  (70.0%)
    Val   :   60 rows  (15.0%)
    Test  :   60 rows  (15.0%)

  Stratification Check (label % per split):
    Label                     |  Train |    Val |   Test
    --------------------------------------------------
    billing_disputes          |  18.6% |  18.3% |  18.3%
    financial_aid             |  18.9% |  18.3% |  18.3%
    hospital_bills      

In [ ]:
#billing and insurance
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    BartForConditionalGeneration, BartTokenizer
)
from rouge_score import rouge_scorer
import numpy as np
import math
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Device ─────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Splits (already in memory) ────────────────────────────────────────────────
# train_bi, val_bi, test_bi

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1 — FROZEN ENCODER + RETRIEVAL INDEX
# ══════════════════════════════════════════════════════════════════════════════

ENC_MODEL     = 'sentence-transformers/all-MiniLM-L6-v2'
enc_tokenizer = AutoTokenizer.from_pretrained(ENC_MODEL)
enc_model     = AutoModel.from_pretrained(ENC_MODEL).to(device)

for param in enc_model.parameters():
    param.requires_grad = False

print(f"Encoder        : {ENC_MODEL}")
print(f"Encoder params : "
      f"{sum(p.numel() for p in enc_model.parameters()):,}  (frozen)")

# ── Frozen Encoder Module ──────────────────────────────────────────────────────
class FrozenEncoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def mean_pool(self, output, attention_mask):
        tokens   = output.last_hidden_state
        mask_exp = attention_mask.unsqueeze(-1).expand(tokens.size()).float()
        return torch.sum(tokens * mask_exp, 1) / \
               torch.clamp(mask_exp.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            output = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )
        return self.mean_pool(output, attention_mask)

encoder = FrozenEncoder(enc_model).to(device)

# ── Encode Helper ──────────────────────────────────────────────────────────────
def encode_texts(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i : i + batch_size]
        encoded = enc_tokenizer(
            batch,
            max_length     = 128,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        embs = encoder(
            encoded['input_ids'].to(device),
            encoded['attention_mask'].to(device)
        )
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

# ── Build Retrieval Index from all splits ─────────────────────────────────────
print("\nBuilding retrieval index...")
full_df       = pd.concat(
    [train_bi, val_bi, test_bi],
    ignore_index=True
).reset_index(drop=True)

full_qa_texts = (full_df['question'] + ' ' + full_df['answer']).tolist()
full_qa_embs  = encode_texts(full_qa_texts)

print(f"Indexed {len(full_df)} Q&A pairs across "
      f"{full_df['topic'].nunique()} topics")
for topic, count in full_df['topic'].value_counts().items():
    print(f"  {topic:25s} : {count} rows")

# ── Retrieval Function ─────────────────────────────────────────────────────────
def retrieve_top_k(query: str, k: int = 3) -> list:
    query_emb = encode_texts([query])
    q_norm    = nn.functional.normalize(
                    torch.tensor(query_emb,     dtype=torch.float32), dim=-1)
    k_norm    = nn.functional.normalize(
                    torch.tensor(full_qa_embs,  dtype=torch.float32), dim=-1)
    sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
    top_k_idx = torch.topk(sims, k=k).indices.tolist()

    return [{
        'question'   : full_df.iloc[i]['question'],
        'answer'     : full_df.iloc[i]['answer'],
        'topic'      : full_df.iloc[i]['topic'],
        'similarity' : round(sims[i].item(), 4)
    } for i in top_k_idx]

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2 — BIOBART COMPOSER
# ══════════════════════════════════════════════════════════════════════════════

BART_MODEL     = 'GanjinZero/biobart-v2-base'
bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL)
bart_model     = BartForConditionalGeneration.from_pretrained(
                     BART_MODEL).to(device)

print(f"\nComposer       : {BART_MODEL}")
print(f"Composer params: "
      f"{sum(p.numel() for p in bart_model.parameters()):,}")

# ── Dataset ────────────────────────────────────────────────────────────────────
class BillingRAGDataset(Dataset):
    def __init__(self, df, retrieve_fn, k=3):
        self.data    = df.reset_index(drop=True)
        self.samples = self._build_samples(retrieve_fn, k)

    def _build_samples(self, retrieve_fn, k):
        samples = []
        for idx in range(len(self.data)):
            row         = self.data.loc[idx]
            context     = retrieve_fn(row['question'], k=k)
            context_str = ' '.join([
                f"Context {i+1}: {c['answer']}"
                for i, c in enumerate(context)
            ])
            samples.append({
                'input'    : (f"Question: {row['question']} "
                              f"{context_str} "
                              f"Compose a helpful hospital billing "
                              f"and insurance answer:"),
                'target'   : row['answer'],
                'topic'    : row['topic'],
                'question' : row['question']
            })
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# ── Collator ───────────────────────────────────────────────────────────────────
class RAGCollator:
    def __init__(self, tokenizer, max_input_len=1024, max_target_len=256):
        self.tokenizer      = tokenizer
        self.max_input_len  = max_input_len
        self.max_target_len = max_target_len

    def __call__(self, batch):
        inputs    = [b['input']    for b in batch]
        targets   = [b['target']   for b in batch]
        topics    = [b['topic']    for b in batch]
        questions = [b['question'] for b in batch]

        inp_enc = self.tokenizer(
            inputs,
            max_length     = self.max_input_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        tgt_enc = self.tokenizer(
            targets,
            max_length     = self.max_target_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )

        labels = tgt_enc['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids'      : inp_enc['input_ids'],
            'attention_mask' : inp_enc['attention_mask'],
            'labels'         : labels,
            'topics'         : topics,
            'questions'      : questions,
            'raw_targets'    : targets
        }

collator = RAGCollator(bart_tokenizer)

# ── DataLoaders ────────────────────────────────────────────────────────────────
print("\nBuilding RAG datasets...")
train_dataset = BillingRAGDataset(train_bi, retrieve_top_k, k=3)
val_dataset   = BillingRAGDataset(val_bi,   retrieve_top_k, k=3)
test_dataset  = BillingRAGDataset(test_bi,  retrieve_top_k, k=3)

train_loader  = DataLoader(train_dataset, batch_size=4,
                            shuffle=True,  collate_fn=collator)
val_loader    = DataLoader(val_dataset,   batch_size=4,
                            shuffle=False, collate_fn=collator)
test_loader   = DataLoader(test_dataset,  batch_size=4,
                            shuffle=False, collate_fn=collator)

print(f"Train : {len(train_dataset)} samples | {len(train_loader)} batches")
print(f"Val   : {len(val_dataset)}   samples | {len(val_loader)}   batches")
print(f"Test  : {len(test_dataset)}  samples | {len(test_loader)}  batches")

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 3 — TRAINING ENGINE
# ══════════════════════════════════════════════════════════════════════════════

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer    = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.min_lr       = min_lr
        self.current_step = 0
        self.base_lrs     = [pg['lr'] for pg in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        for pg, lr in zip(self.optimizer.param_groups, self._compute_lr()):
            pg['lr'] = lr

    def _compute_lr(self):
        s = self.current_step
        lrs = []
        for base_lr in self.base_lrs:
            if s < self.warmup_steps:
                lr = base_lr * (s / max(1, self.warmup_steps))
            else:
                progress = (s - self.warmup_steps) / \
                           max(1, self.total_steps - self.warmup_steps)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * \
                     (1 + math.cos(math.pi * progress))
            lrs.append(lr)
        return lrs

class BillingRAGTrainer:
    def __init__(self, model, optimizer, scheduler, tokenizer,
                 device, accum_steps=4, max_grad_norm=1.0,
                 checkpoint_path='best_billing_rag.pt'):
        self.model           = model
        self.optimizer       = optimizer
        self.scheduler       = scheduler
        self.tokenizer       = tokenizer
        self.device          = device
        self.accum_steps     = accum_steps
        self.max_grad_norm   = max_grad_norm
        self.checkpoint_path = checkpoint_path
        self.best_val_loss   = float('inf')
        self.patience_count  = 0

    def train_epoch(self, loader):
        self.model.train()
        total_loss, total_tokens = 0.0, 0
        self.optimizer.zero_grad()

        for step, batch in enumerate(loader):
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels         = batch['labels'].to(self.device)

            outputs        = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels
            )
            loss           = outputs.loss / self.accum_steps
            loss.backward()

            n_tokens       = (labels != -100).sum().item()
            total_loss    += outputs.loss.item() * n_tokens
            total_tokens  += n_tokens

            if (step + 1) % self.accum_steps == 0 or \
               (step + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.max_grad_norm
                )
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()

        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity

    def evaluate(self, loader):
        self.model.eval()
        total_loss, total_tokens  = 0.0, 0
        all_preds, all_refs       = [], []
        all_topics, all_questions = [], []

        with torch.no_grad():
            for batch in loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['labels'].to(self.device)

                outputs        = self.model(
                    input_ids      = input_ids,
                    attention_mask = attention_mask,
                    labels         = labels
                )
                n_tokens       = (labels != -100).sum().item()
                total_loss    += outputs.loss.item() * n_tokens
                total_tokens  += n_tokens

                generated = self.model.generate(
                    input_ids            = input_ids,
                    attention_mask       = attention_mask,
                    max_new_tokens       = 256,
                    num_beams            = 4,
                    early_stopping       = True,
                    no_repeat_ngram_size = 3,
                    length_penalty       = 1.0
                )

                preds     = self.tokenizer.batch_decode(
                    generated, skip_special_tokens=True
                )
                label_ids = labels.clone()
                label_ids[label_ids == -100] = self.tokenizer.pad_token_id
                refs      = self.tokenizer.batch_decode(
                    label_ids, skip_special_tokens=True
                )

                all_preds.extend(preds)
                all_refs.extend(refs)
                all_topics.extend(batch['topics'])
                all_questions.extend(batch['questions'])

        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity, all_preds, all_refs, \
               all_topics, all_questions

    def check_early_stop(self, val_loss, patience=3):
        if val_loss < self.best_val_loss:
            self.best_val_loss  = val_loss
            self.patience_count = 0
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : val_loss
            }, self.checkpoint_path)
            return False
        self.patience_count += 1
        return self.patience_count >= patience

    def load_best(self):
        if not os.path.exists(self.checkpoint_path):
            print("⚠️  No checkpoint found — saving current state...")
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : self.best_val_loss
            }, self.checkpoint_path)
        ckpt = torch.load(self.checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt['model_state'])
        print(f"✅ Best checkpoint restored — "
              f"Val Loss: {ckpt['val_loss']:.4f}")

# ── Metrics ────────────────────────────────────────────────────────────────────
rouge_scorer_obj = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
)

def compute_rouge(preds, refs):
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = rouge_scorer_obj.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    return {
        'rouge1' : round(np.mean(r1), 4),
        'rouge2' : round(np.mean(r2), 4),
        'rougeL' : round(np.mean(rL), 4)
    }

def compute_token_f1(pred, ref):
    pred_tokens = pred.lower().split()
    ref_tokens  = ref.lower().split()
    common      = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    p = len(common) / len(pred_tokens) if pred_tokens else 0
    r = len(common) / len(ref_tokens)  if ref_tokens  else 0
    return round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0.0

def print_metrics(split_name, loss, ppl, preds, refs, topics, questions):
    rouge     = compute_rouge(preds, refs)
    token_f1s = [compute_token_f1(p, r) for p, r in zip(preds, refs)]
    em        = sum(p.strip().lower() == r.strip().lower()
                    for p, r in zip(preds, refs)) / len(preds)

    print(f"\n{'='*60}")
    print(f"  Metrics — {split_name}")
    print(f"{'='*60}")
    print(f"  Loss        : {loss:.4f}")
    print(f"  Perplexity  : {ppl:.4f}")
    print(f"  ROUGE-1     : {rouge['rouge1']:.4f}")
    print(f"  ROUGE-2     : {rouge['rouge2']:.4f}")
    print(f"  ROUGE-L     : {rouge['rougeL']:.4f}")
    print(f"  Token F1    : {np.mean(token_f1s):.4f}")
    print(f"  Exact Match : {em:.4f}")
    print(f"\n  Per-Topic ROUGE-L:")
    for topic in sorted(set(topics)):
        mask    = [t == topic for t in topics]
        t_preds = [p for p, m in zip(preds, mask) if m]
        t_refs  = [r for r, m in zip(refs,  mask) if m]
        t_rouge = compute_rouge(t_preds, t_refs)
        t_f1    = np.mean([compute_token_f1(p, r)
                           for p, r in zip(t_preds, t_refs)])
        print(f"    {topic:25s} : ROUGE-L {t_rouge['rougeL']:.4f} | "
              f"Token F1 {t_f1:.4f} | n={len(t_preds)}")

# ── Initialise ─────────────────────────────────────────────────────────────────
EPOCHS       = 15
ACCUM_STEPS  = 4
total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = total_steps // 10

optimizer = torch.optim.AdamW(
    bart_model.parameters(), lr=3e-5, weight_decay=0.01
)
scheduler = WarmupCosineScheduler(optimizer, warmup_steps, total_steps)
trainer   = BillingRAGTrainer(
    bart_model, optimizer, scheduler,
    bart_tokenizer, device,
    accum_steps     = ACCUM_STEPS,
    checkpoint_path = 'best_billing_rag.pt'
)

# ── Training Loop ──────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  Training Billing BioBART RAG  |  {EPOCHS} epochs  |  "
      f"Warmup {warmup_steps}  |  Total {total_steps} steps")
print(f"{'='*70}")
print(f"  {'Epoch':>5} | {'Train Loss':>10} | {'Train PPL':>9} | "
      f"{'Val Loss':>8} | {'Val PPL':>7} | {'ROUGE-L':>7} | Status")
print(f"  {'-'*70}")

for epoch in range(EPOCHS):
    tr_loss, tr_ppl       = trainer.train_epoch(train_loader)
    vl_loss, vl_ppl, \
    vl_preds, vl_refs, \
    vl_topics, vl_qs      = trainer.evaluate(val_loader)
    vl_rouge              = compute_rouge(vl_preds, vl_refs)
    stop                  = trainer.check_early_stop(vl_loss, patience=3)
    status                = "← best ✅" if trainer.patience_count == 0 \
                            else f"patience {trainer.patience_count}/3"

    print(f"  {epoch+1:>5} | {tr_loss:>10.4f} | {tr_ppl:>9.4f} | "
          f"{vl_loss:>8.4f} | {vl_ppl:>7.4f} | "
          f"{vl_rouge['rougeL']:>7.4f} | {status}")

    if stop:
        print(f"\n  Early stopping at epoch {epoch+1}")
        break

# ══════════════════════════════════════════════════════════════════════════════
#  METRICS — ALL SPLITS + SAMPLE PREDICTIONS
# ══════════════════════════════════════════════════════════════════════════════

trainer.load_best()

tr_loss, tr_ppl, tr_preds, \
tr_refs, tr_topics, tr_qs  = trainer.evaluate(train_loader)

vl_loss, vl_ppl, vl_preds, \
vl_refs, vl_topics, vl_qs  = trainer.evaluate(val_loader)

ts_loss, ts_ppl, ts_preds, \
ts_refs, ts_topics, ts_qs  = trainer.evaluate(test_loader)

print_metrics("Train",      tr_loss, tr_ppl,
              tr_preds, tr_refs, tr_topics, tr_qs)
print_metrics("Validation", vl_loss, vl_ppl,
              vl_preds, vl_refs, vl_topics, vl_qs)
print_metrics("Test",       ts_loss, ts_ppl,
              ts_preds, ts_refs, ts_topics, ts_qs)

# ── Sample Predictions from Test Set ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Sample Predictions — Test Set (5 random)")
print(f"{'='*60}")

for i in np.random.choice(len(ts_preds), 5, replace=False):
    r = compute_rouge([ts_preds[i]], [ts_refs[i]])
    f = compute_token_f1(ts_preds[i], ts_refs[i])
    print(f"\n  Topic     : {ts_topics[i]}")
    print(f"  Question  : {ts_qs[i]}")
    print(f"  Reference : {ts_refs[i][:150]}...")
    print(f"  Generated : {ts_preds[i][:150]}...")
    print(f"  ROUGE-L   : {r['rougeL']:.4f} | Token F1 : {f:.4f}")
    print(f"  {'-'*55}")

Device : cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder        : sentence-transformers/all-MiniLM-L6-v2
Encoder params : 22,713,216  (frozen)

Building retrieval index...
Indexed 400 Q&A pairs across 5 topics
  hospital_bills            : 85 rows
  insurance_claims          : 85 rows
  insurance_policy          : 81 rows
  financial_aid             : 75 rows
  billing_disputes          : 74 rows


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


Composer       : GanjinZero/biobart-v2-base
Composer params: 166,404,864

Building RAG datasets...
Train : 280 samples | 70 batches
Val   : 60   samples | 15   batches
Test  : 60  samples | 15  batches

  Training Billing BioBART RAG  |  15 epochs  |  Warmup 25  |  Total 255 steps
  Epoch | Train Loss | Train PPL | Val Loss | Val PPL | ROUGE-L | Status
  ----------------------------------------------------------------------
      1 |     0.8837 |    2.4198 |   0.0795 |  1.0827 |  0.9908 | ← best ✅
      2 |     0.0249 |    1.0252 |   0.0243 |  1.0246 |  0.9966 | ← best ✅
      3 |     0.0066 |    1.0066 |   0.0238 |  1.0241 |  0.9973 | ← best ✅
      4 |     0.0021 |    1.0021 |   0.0222 |  1.0224 |  0.9967 | ← best ✅
      5 |     0.0014 |    1.0014 |   0.0231 |  1.0234 |  0.9959 | patience 1/3
      6 |     0.0006 |    1.0006 |   0.0239 |  1.0242 |  0.9945 | patience 2/3
      7 |     0.0006 |    1.0006 |   0.0245 |  1.0248 |  0.9953 | patience 3/3

  Early stopping at epoch 7
✅ Bes

In [ ]:
# ── Save Billing and Insurance Model Files ────────────────────────────────────
import os
import numpy as np
from google.colab import files

os.makedirs('saved_models/billing', exist_ok=True)

# ── Save 1: Trained BioBART Weights ───────────────────────────────────────────
torch.save(
    bart_model.state_dict(),
    'saved_models/billing/billing_biobart_weights.pt'
)
print("✅ Saved: billing_biobart_weights.pt")

# ── Save 2: Retrieval Embeddings ───────────────────────────────────────────────
np.save(
    'saved_models/billing/billing_qa_embeddings.npy',
    full_qa_embs
)
print("✅ Saved: billing_qa_embeddings.npy")

# ── Save 3: Q&A Data ───────────────────────────────────────────────────────────
full_df.to_csv(
    'saved_models/billing/billing_qa_data.csv',
    index=False
)
print("✅ Saved: billing_qa_data.csv")

# ── Verify Everything Before Downloading ──────────────────────────────────────
print("\n── Verification ──────────────────────────────────────────────────────")

# Check 1: weights file size
weights_size = os.path.getsize(
    'saved_models/billing/billing_biobart_weights.pt'
)
print(f"   billing_biobart_weights.pt  : {weights_size / 1e6:.1f} MB")

# Check 2: embeddings shape
test_emb_load = np.load('saved_models/billing/billing_qa_embeddings.npy')
print(f"   billing_qa_embeddings.npy   : shape {test_emb_load.shape}")

# Check 3: csv rows
test_csv_load = pd.read_csv('saved_models/billing/billing_qa_data.csv')
print(f"   billing_qa_data.csv         : {len(test_csv_load)} rows, "
      f"columns: {list(test_csv_load.columns)}")

# Check 4: embeddings rows must match csv rows
assert test_emb_load.shape[0] == len(test_csv_load), \
    f"❌ MISMATCH — embeddings {test_emb_load.shape[0]} rows " \
    f"vs csv {len(test_csv_load)} rows"
print(f"\n✅ Verification passed — embeddings and csv rows match perfectly")

# ── Quick End-to-End Inference Test Before Downloading ────────────────────────
print("\n── Quick Inference Test ──────────────────────────────────────────────")

from transformers import BartForConditionalGeneration, BartTokenizer
import torch.nn.functional as F

# Reload weights into a fresh model instance to simulate PyCharm loading
test_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
test_model.load_state_dict(
    torch.load(
        'saved_models/billing/billing_biobart_weights.pt',
        map_location=device
    )
)
test_model.to(device)
test_model.eval()

# Run one test query through the full pipeline
test_query     = "How do I claim insurance reimbursement at AIIMS?"
test_query_emb = encode_texts([test_query])

# Retrieve top 3
q_norm    = F.normalize(
    torch.tensor(test_query_emb, dtype=torch.float32), dim=-1
)
k_norm    = F.normalize(
    torch.tensor(test_emb_load,  dtype=torch.float32), dim=-1
)
sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
top_k_idx = torch.topk(sims, k=3).indices.tolist()

context_str = ' '.join([
    f"Context {i+1}: {test_csv_load.iloc[idx]['answer']}"
    for i, idx in enumerate(top_k_idx)
])

input_text = (
    f"Question: {test_query} "
    f"{context_str} "
    f"Compose a helpful hospital billing and insurance answer:"
)

inputs = bart_tokenizer(
    input_text,
    max_length     = 1024,
    truncation     = True,
    return_tensors = 'pt'
).to(device)

with torch.no_grad():
    output = test_model.generate(
        input_ids            = inputs['input_ids'],
        attention_mask       = inputs['attention_mask'],
        max_new_tokens       = 256,
        num_beams            = 4,
        early_stopping       = True,
        no_repeat_ngram_size = 3,
        length_penalty       = 1.0
    )

test_answer = bart_tokenizer.decode(output[0], skip_special_tokens=True)
print(f"   Query  : {test_query}")
print(f"   Answer : {test_answer[:200]}...")
print(f"\n✅ Inference test passed — model loads and generates correctly")

# ── Download All Three Files ───────────────────────────────────────────────────
print("\n── Downloading Files ─────────────────────────────────────────────────")
files.download('saved_models/billing/billing_biobart_weights.pt')
files.download('saved_models/billing/billing_qa_embeddings.npy')
files.download('saved_models/billing/billing_qa_data.csv')

print("\n✅ All 3 Billing files downloaded.")
print("   → billing_biobart_weights.pt")
print("   → billing_qa_embeddings.npy")
print("   → billing_qa_data.csv")

✅ Saved: billing_biobart_weights.pt
✅ Saved: billing_qa_embeddings.npy
✅ Saved: billing_qa_data.csv

── Verification ──────────────────────────────────────────────────────
   billing_biobart_weights.pt  : 666.1 MB
   billing_qa_embeddings.npy   : shape (400, 384)
   billing_qa_data.csv         : 400 rows, columns: ['question', 'answer', 'topic']

✅ Verification passed — embeddings and csv rows match perfectly

── Quick Inference Test ──────────────────────────────────────────────


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

   Query  : How do I claim insurance reimbursement at AIIMS?
   Answer : Request a written explanation of the rejection from your insurer, review the policy terms, and submit a formal appeal with any additional supporting documentation....

✅ Inference test passed — model loads and generates correctly

── Downloading Files ─────────────────────────────────────────────────


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All 3 Billing files downloaded.
   → billing_biobart_weights.pt
   → billing_qa_embeddings.npy
   → billing_qa_data.csv


In [ ]:
train_da, val_da, test_da = prepare_model_dataset(df_da)


  Dataset Preparation
  Raw shape        : (400, 3)
  Clean shape      : (400, 5)
  Labels           : ['booking', 'cancellation', 'consultation', 'doctor_info', 'practicalities', 'referral', 'results']

  Samples per label:
    booking                   :   55  ███████████
    cancellation              :   48  █████████
    consultation              :   56  ███████████
    doctor_info               :   57  ███████████
    practicalities            :   78  ███████████████
    referral                  :   51  ██████████
    results                   :   55  ███████████

  Question length  : mean=11.8 | max=20 | min=6
  Answer length    : mean=22.8 | max=32 | min=15

  Split Summary:
    Train :  280 rows  (70.0%)
    Val   :   60 rows  (15.0%)
    Test  :   60 rows  (15.0%)

  Stratification Check (label % per split):
    Label                     |  Train |    Val |   Test
    --------------------------------------------------
    booking                   |  13.6% |  15.0% |  13.3%


In [ ]:
# doctor appointment - COMPLETE FIXED VERSION
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    BartForConditionalGeneration, BartTokenizer
)
from rouge_score import rouge_scorer
import numpy as np
import math
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Device ─────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1 — FROZEN ENCODER + RETRIEVAL INDEX
# ══════════════════════════════════════════════════════════════════════════════

ENC_MODEL     = 'sentence-transformers/all-MiniLM-L6-v2'
enc_tokenizer = AutoTokenizer.from_pretrained(ENC_MODEL)
enc_model     = AutoModel.from_pretrained(ENC_MODEL).to(device)

for param in enc_model.parameters():
    param.requires_grad = False

print(f"Encoder        : {ENC_MODEL}")
print(f"Encoder params : "
      f"{sum(p.numel() for p in enc_model.parameters()):,}  (frozen)")

# ── Frozen Encoder Module ──────────────────────────────────────────────────────
class FrozenEncoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def mean_pool(self, output, attention_mask):
        tokens   = output.last_hidden_state
        mask_exp = attention_mask.unsqueeze(-1).expand(tokens.size()).float()
        return torch.sum(tokens * mask_exp, 1) / \
               torch.clamp(mask_exp.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            output = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )
        return self.mean_pool(output, attention_mask)

encoder = FrozenEncoder(enc_model).to(device)

# ── Encode Helper ──────────────────────────────────────────────────────────────
def encode_texts(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i : i + batch_size]
        encoded = enc_tokenizer(
            batch,
            max_length     = 128,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        embs = encoder(
            encoded['input_ids'].to(device),
            encoded['attention_mask'].to(device)
        )
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

# ── Build Retrieval Index ──────────────────────────────────────────────────────
print("\nBuilding retrieval index...")
full_df       = pd.concat(
    [train_da, val_da, test_da],
    ignore_index=True
).reset_index(drop=True)

full_qa_texts = (full_df['question'] + ' ' + full_df['answer']).tolist()
full_qa_embs  = encode_texts(full_qa_texts)

print(f"Indexed {len(full_df)} Q&A pairs across "
      f"{full_df['topic'].nunique()} topics")
for topic, count in full_df['topic'].value_counts().items():
    print(f"  {topic:25s} : {count} rows")

# ── Retrieval Function ─────────────────────────────────────────────────────────
def retrieve_top_k(query: str, k: int = 3) -> list:
    query_emb = encode_texts([query])
    q_norm    = nn.functional.normalize(
                    torch.tensor(query_emb,    dtype=torch.float32), dim=-1)
    k_norm    = nn.functional.normalize(
                    torch.tensor(full_qa_embs, dtype=torch.float32), dim=-1)
    sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
    top_k_idx = torch.topk(sims, k=k).indices.tolist()

    return [{
        'question'   : full_df.iloc[i]['question'],
        'answer'     : full_df.iloc[i]['answer'],
        'topic'      : full_df.iloc[i]['topic'],
        'similarity' : round(sims[i].item(), 4)
    } for i in top_k_idx]

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2 — BIOBART COMPOSER
# ══════════════════════════════════════════════════════════════════════════════

BART_MODEL     = 'GanjinZero/biobart-v2-base'
bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL)
bart_model     = BartForConditionalGeneration.from_pretrained(
                     BART_MODEL).to(device)

print(f"\nComposer       : {BART_MODEL}")
print(f"Composer params: "
      f"{sum(p.numel() for p in bart_model.parameters()):,}")

# ── Dataset ────────────────────────────────────────────────────────────────────
class DoctorAppointmentRAGDataset(Dataset):
    def __init__(self, df, retrieve_fn, k=3):
        self.data    = df.reset_index(drop=True)
        self.samples = self._build_samples(retrieve_fn, k)

    def _build_samples(self, retrieve_fn, k):
        samples = []
        for idx in range(len(self.data)):
            row         = self.data.loc[idx]
            context     = retrieve_fn(row['question'], k=k)
            context_str = ' '.join([
                f"Context {i+1}: {c['answer']}"
                for i, c in enumerate(context)
            ])
            samples.append({
                'input'    : (f"Question: {row['question']} "
                              f"{context_str} "
                              f"Compose a helpful doctor appointment answer:"),
                'target'   : row['answer'],
                'topic'    : row['topic'],
                'question' : row['question']
            })
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# ── Collator ───────────────────────────────────────────────────────────────────
class RAGCollator:
    def __init__(self, tokenizer, max_input_len=1024, max_target_len=256):
        self.tokenizer      = tokenizer
        self.max_input_len  = max_input_len
        self.max_target_len = max_target_len

    def __call__(self, batch):
        inputs    = [b['input']    for b in batch]
        targets   = [b['target']   for b in batch]
        topics    = [b['topic']    for b in batch]
        questions = [b['question'] for b in batch]

        inp_enc = self.tokenizer(
            inputs,
            max_length     = self.max_input_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        tgt_enc = self.tokenizer(
            targets,
            max_length     = self.max_target_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )

        labels = tgt_enc['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids'      : inp_enc['input_ids'],
            'attention_mask' : inp_enc['attention_mask'],
            'labels'         : labels,
            'topics'         : topics,
            'questions'      : questions,
            'raw_targets'    : targets
        }

collator = RAGCollator(bart_tokenizer)

# ── DataLoaders ────────────────────────────────────────────────────────────────
print("\nBuilding RAG datasets...")
train_dataset = DoctorAppointmentRAGDataset(train_da, retrieve_top_k, k=3)
val_dataset   = DoctorAppointmentRAGDataset(val_da,   retrieve_top_k, k=3)
test_dataset  = DoctorAppointmentRAGDataset(test_da,  retrieve_top_k, k=3)

train_loader  = DataLoader(train_dataset, batch_size=4,
                            shuffle=True,  collate_fn=collator)
val_loader    = DataLoader(val_dataset,   batch_size=4,
                            shuffle=False, collate_fn=collator)
test_loader   = DataLoader(test_dataset,  batch_size=4,
                            shuffle=False, collate_fn=collator)

print(f"Train : {len(train_dataset)} samples | {len(train_loader)} batches")
print(f"Val   : {len(val_dataset)}   samples | {len(val_loader)}   batches")
print(f"Test  : {len(test_dataset)}  samples | {len(test_loader)}  batches")

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 3 — TRAINING ENGINE
# ══════════════════════════════════════════════════════════════════════════════

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer    = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.min_lr       = min_lr
        self.current_step = 0
        self.base_lrs     = [pg['lr'] for pg in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        for pg, lr in zip(self.optimizer.param_groups, self._compute_lr()):
            pg['lr'] = lr

    def _compute_lr(self):
        s = self.current_step
        lrs = []
        for base_lr in self.base_lrs:
            if s < self.warmup_steps:
                lr = base_lr * (s / max(1, self.warmup_steps))
            else:
                progress = (s - self.warmup_steps) / \
                           max(1, self.total_steps - self.warmup_steps)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * \
                     (1 + math.cos(math.pi * progress))
            lrs.append(lr)
        return lrs

class DoctorAppointmentRAGTrainer:
    def __init__(self, model, optimizer, scheduler, tokenizer,
                 device, accum_steps=4, max_grad_norm=1.0,
                 checkpoint_path='best_da_rag.pt'):
        self.model           = model
        self.optimizer       = optimizer
        self.scheduler       = scheduler
        self.tokenizer       = tokenizer
        self.device          = device
        self.accum_steps     = accum_steps
        self.max_grad_norm   = max_grad_norm
        self.checkpoint_path = checkpoint_path
        self.best_val_loss   = float('inf')
        self.patience_count  = 0

    def train_epoch(self, loader):
        self.model.train()
        total_loss, total_tokens = 0.0, 0
        self.optimizer.zero_grad()

        for step, batch in enumerate(loader):
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels         = batch['labels'].to(self.device)

            outputs        = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels
            )
            loss           = outputs.loss / self.accum_steps
            loss.backward()

            n_tokens       = (labels != -100).sum().item()
            total_loss    += outputs.loss.item() * n_tokens
            total_tokens  += n_tokens

            if (step + 1) % self.accum_steps == 0 or \
               (step + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.max_grad_norm
                )
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()

        # ── THIS LINE WAS MISSING IN YOUR ORIGINAL ────────────────────────────
        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity

    def evaluate(self, loader):
        self.model.eval()
        total_loss, total_tokens  = 0.0, 0
        all_preds, all_refs       = [], []
        all_topics, all_questions = [], []

        with torch.no_grad():
            for batch in loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['labels'].to(self.device)

                outputs        = self.model(
                    input_ids      = input_ids,
                    attention_mask = attention_mask,
                    labels         = labels
                )
                n_tokens       = (labels != -100).sum().item()
                total_loss    += outputs.loss.item() * n_tokens
                total_tokens  += n_tokens

                generated = self.model.generate(
                    input_ids            = input_ids,
                    attention_mask       = attention_mask,
                    max_new_tokens       = 256,
                    num_beams            = 4,
                    early_stopping       = True,
                    no_repeat_ngram_size = 3,
                    length_penalty       = 1.0
                )

                preds     = self.tokenizer.batch_decode(
                    generated, skip_special_tokens=True
                )
                label_ids = labels.clone()
                label_ids[label_ids == -100] = self.tokenizer.pad_token_id
                refs      = self.tokenizer.batch_decode(
                    label_ids, skip_special_tokens=True
                )

                all_preds.extend(preds)
                all_refs.extend(refs)
                all_topics.extend(batch['topics'])
                all_questions.extend(batch['questions'])

        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity, all_preds, all_refs, \
               all_topics, all_questions

    def check_early_stop(self, val_loss, patience=3):
        if val_loss < self.best_val_loss:
            self.best_val_loss  = val_loss
            self.patience_count = 0
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : val_loss
            }, self.checkpoint_path)
            return False
        self.patience_count += 1
        return self.patience_count >= patience

    def load_best(self):
        if not os.path.exists(self.checkpoint_path):
            print("⚠️  No checkpoint found — saving current state...")
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : self.best_val_loss
            }, self.checkpoint_path)
        ckpt = torch.load(self.checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt['model_state'])
        print(f"✅ Best checkpoint restored — "
              f"Val Loss: {ckpt['val_loss']:.4f}")

# ── Metrics ────────────────────────────────────────────────────────────────────
rouge_scorer_obj = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
)

def compute_rouge(preds, refs):
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = rouge_scorer_obj.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    return {
        'rouge1' : round(np.mean(r1), 4),
        'rouge2' : round(np.mean(r2), 4),
        'rougeL' : round(np.mean(rL), 4)
    }

def compute_token_f1(pred, ref):
    pred_tokens = pred.lower().split()
    ref_tokens  = ref.lower().split()
    common      = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    p = len(common) / len(pred_tokens) if pred_tokens else 0
    r = len(common) / len(ref_tokens)  if ref_tokens  else 0
    return round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0.0

def print_metrics(split_name, loss, ppl, preds, refs, topics, questions):
    rouge     = compute_rouge(preds, refs)
    token_f1s = [compute_token_f1(p, r) for p, r in zip(preds, refs)]
    em        = sum(p.strip().lower() == r.strip().lower()
                    for p, r in zip(preds, refs)) / len(preds)

    print(f"\n{'='*60}")
    print(f"  Metrics — {split_name}")
    print(f"{'='*60}")
    print(f"  Loss        : {loss:.4f}")
    print(f"  Perplexity  : {ppl:.4f}")
    print(f"  ROUGE-1     : {rouge['rouge1']:.4f}")
    print(f"  ROUGE-2     : {rouge['rouge2']:.4f}")
    print(f"  ROUGE-L     : {rouge['rougeL']:.4f}")
    print(f"  Token F1    : {np.mean(token_f1s):.4f}")
    print(f"  Exact Match : {em:.4f}")
    print(f"\n  Per-Topic ROUGE-L:")
    for topic in sorted(set(topics)):
        mask    = [t == topic for t in topics]
        t_preds = [p for p, m in zip(preds, mask) if m]
        t_refs  = [r for r, m in zip(refs,  mask) if m]
        t_rouge = compute_rouge(t_preds, t_refs)
        t_f1    = np.mean([compute_token_f1(p, r)
                           for p, r in zip(t_preds, t_refs)])
        print(f"    {topic:25s} : ROUGE-L {t_rouge['rougeL']:.4f} | "
              f"Token F1 {t_f1:.4f} | n={len(t_preds)}")

# ── Initialise ─────────────────────────────────────────────────────────────────
EPOCHS       = 15
ACCUM_STEPS  = 4
total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = total_steps // 10

optimizer = torch.optim.AdamW(
    bart_model.parameters(), lr=3e-5, weight_decay=0.01
)
scheduler = WarmupCosineScheduler(optimizer, warmup_steps, total_steps)
trainer   = DoctorAppointmentRAGTrainer(
    bart_model, optimizer, scheduler,
    bart_tokenizer, device,
    accum_steps     = ACCUM_STEPS,
    checkpoint_path = 'best_da_rag.pt'
)

# ── Training Loop ──────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  Training Doctor Appointment BioBART RAG  |  {EPOCHS} epochs  |  "
      f"Warmup {warmup_steps}  |  Total {total_steps} steps")
print(f"{'='*70}")
print(f"  {'Epoch':>5} | {'Train Loss':>10} | {'Train PPL':>9} | "
      f"{'Val Loss':>8} | {'Val PPL':>7} | {'ROUGE-L':>7} | Status")
print(f"  {'-'*70}")

for epoch in range(EPOCHS):
    tr_loss, tr_ppl       = trainer.train_epoch(train_loader)
    vl_loss, vl_ppl, \
    vl_preds, vl_refs, \
    vl_topics, vl_qs      = trainer.evaluate(val_loader)
    vl_rouge              = compute_rouge(vl_preds, vl_refs)
    stop                  = trainer.check_early_stop(vl_loss, patience=3)
    status                = "← best ✅" if trainer.patience_count == 0 \
                            else f"patience {trainer.patience_count}/3"

    print(f"  {epoch+1:>5} | {tr_loss:>10.4f} | {tr_ppl:>9.4f} | "
          f"{vl_loss:>8.4f} | {vl_ppl:>7.4f} | "
          f"{vl_rouge['rougeL']:>7.4f} | {status}")

    if stop:
        print(f"\n  Early stopping at epoch {epoch+1}")
        break

# ══════════════════════════════════════════════════════════════════════════════
#  METRICS — ALL SPLITS + SAMPLE PREDICTIONS
# ══════════════════════════════════════════════════════════════════════════════

trainer.load_best()

tr_loss, tr_ppl, tr_preds, \
tr_refs, tr_topics, tr_qs  = trainer.evaluate(train_loader)

vl_loss, vl_ppl, vl_preds, \
vl_refs, vl_topics, vl_qs  = trainer.evaluate(val_loader)

ts_loss, ts_ppl, ts_preds, \
ts_refs, ts_topics, ts_qs  = trainer.evaluate(test_loader)

print_metrics("Train",      tr_loss, tr_ppl,
              tr_preds, tr_refs, tr_topics, tr_qs)
print_metrics("Validation", vl_loss, vl_ppl,
              vl_preds, vl_refs, vl_topics, vl_qs)
print_metrics("Test",       ts_loss, ts_ppl,
              ts_preds, ts_refs, ts_topics, ts_qs)

# ── Sample Predictions from Test Set ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Sample Predictions — Test Set (5 random)")
print(f"{'='*60}")

for i in np.random.choice(len(ts_preds), 5, replace=False):
    r = compute_rouge([ts_preds[i]], [ts_refs[i]])
    f = compute_token_f1(ts_preds[i], ts_refs[i])
    print(f"\n  Topic     : {ts_topics[i]}")
    print(f"  Question  : {ts_qs[i]}")
    print(f"  Reference : {ts_refs[i][:150]}...")
    print(f"  Generated : {ts_preds[i][:150]}...")
    print(f"  ROUGE-L   : {r['rougeL']:.4f} | Token F1 : {f:.4f}")
    print(f"  {'-'*55}")

Device : cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder        : sentence-transformers/all-MiniLM-L6-v2
Encoder params : 22,713,216  (frozen)

Building retrieval index...
Indexed 400 Q&A pairs across 7 topics
  practicalities            : 78 rows
  doctor_info               : 57 rows
  consultation              : 56 rows
  booking                   : 55 rows
  results                   : 55 rows
  referral                  : 51 rows
  cancellation              : 48 rows


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


Composer       : GanjinZero/biobart-v2-base
Composer params: 166,404,864

Building RAG datasets...
Train : 280 samples | 70 batches
Val   : 60   samples | 15   batches
Test  : 60  samples | 15  batches

  Training Doctor Appointment BioBART RAG  |  15 epochs  |  Warmup 25  |  Total 255 steps
  Epoch | Train Loss | Train PPL | Val Loss | Val PPL | ROUGE-L | Status
  ----------------------------------------------------------------------
      1 |     0.9943 |    2.7029 |   0.0796 |  1.0828 |  0.9996 | ← best ✅
      2 |     0.0485 |    1.0497 |   0.0107 |  1.0107 |  0.9993 | ← best ✅
      3 |     0.0098 |    1.0098 |   0.0100 |  1.0101 |  0.9985 | ← best ✅
      4 |     0.0061 |    1.0061 |   0.0088 |  1.0089 |  0.9985 | ← best ✅
      5 |     0.0069 |    1.0069 |   0.0093 |  1.0093 |  0.9985 | patience 1/3
      6 |     0.0031 |    1.0031 |   0.0092 |  1.0092 |  0.9985 | patience 2/3
      7 |     0.0010 |    1.0010 |   0.0093 |  1.0093 |  0.9985 | patience 3/3

  Early stopping at ep

In [ ]:
# ── Save Doctor Appointment Model Files ───────────────────────────────────────
import os
import numpy as np
from google.colab import files

os.makedirs('saved_models/doctor_appointment', exist_ok=True)

# ── Save 1: Trained BioBART Weights ───────────────────────────────────────────
torch.save(
    bart_model.state_dict(),
    'saved_models/doctor_appointment/da_biobart_weights.pt'
)
print("✅ Saved: da_biobart_weights.pt")

# ── Save 2: Retrieval Embeddings ───────────────────────────────────────────────
np.save(
    'saved_models/doctor_appointment/da_qa_embeddings.npy',
    full_qa_embs
)
print("✅ Saved: da_qa_embeddings.npy")

# ── Save 3: Q&A Data ───────────────────────────────────────────────────────────
full_df.to_csv(
    'saved_models/doctor_appointment/da_qa_data.csv',
    index=False
)
print("✅ Saved: da_qa_data.csv")

# ── Verify Everything Before Downloading ──────────────────────────────────────
print("\n── Verification ──────────────────────────────────────────────────────")

weights_size = os.path.getsize(
    'saved_models/doctor_appointment/da_biobart_weights.pt'
)
print(f"   da_biobart_weights.pt  : {weights_size / 1e6:.1f} MB")

test_emb_load = np.load(
    'saved_models/doctor_appointment/da_qa_embeddings.npy'
)
print(f"   da_qa_embeddings.npy   : shape {test_emb_load.shape}")

test_csv_load = pd.read_csv(
    'saved_models/doctor_appointment/da_qa_data.csv'
)
print(f"   da_qa_data.csv         : {len(test_csv_load)} rows, "
      f"columns: {list(test_csv_load.columns)}")

assert test_emb_load.shape[0] == len(test_csv_load), \
    f"❌ MISMATCH — embeddings {test_emb_load.shape[0]} rows " \
    f"vs csv {len(test_csv_load)} rows"
print(f"\n✅ Verification passed — embeddings and csv rows match perfectly")

# ── Quick Inference Test ───────────────────────────────────────────────────────
print("\n── Quick Inference Test ──────────────────────────────────────────────")

import torch.nn.functional as F
from transformers import BartForConditionalGeneration, BartTokenizer

test_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
test_model.load_state_dict(
    torch.load(
        'saved_models/doctor_appointment/da_biobart_weights.pt',
        map_location=device
    )
)
test_model.to(device)
test_model.eval()

test_query     = "How do I book an appointment with a specialist at AIIMS?"
test_query_emb = encode_texts([test_query])

q_norm    = F.normalize(
    torch.tensor(test_query_emb, dtype=torch.float32), dim=-1
)
k_norm    = F.normalize(
    torch.tensor(test_emb_load,  dtype=torch.float32), dim=-1
)
sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
top_k_idx = torch.topk(sims, k=3).indices.tolist()

context_str = ' '.join([
    f"Context {i+1}: {test_csv_load.iloc[idx]['answer']}"
    for i, idx in enumerate(top_k_idx)
])

input_text = (
    f"Question: {test_query} "
    f"{context_str} "
    f"Compose a helpful doctor appointment answer:"
)

inputs = bart_tokenizer(
    input_text,
    max_length     = 1024,
    truncation     = True,
    return_tensors = 'pt'
).to(device)

with torch.no_grad():
    output = test_model.generate(
        input_ids            = inputs['input_ids'],
        attention_mask       = inputs['attention_mask'],
        max_new_tokens       = 256,
        num_beams            = 4,
        early_stopping       = True,
        no_repeat_ngram_size = 3,
        length_penalty       = 1.0
    )

test_answer = bart_tokenizer.decode(output[0], skip_special_tokens=True)
print(f"   Query  : {test_query}")
print(f"   Answer : {test_answer[:200]}...")
print(f"\n✅ Inference test passed — model loads and generates correctly")

# ── Download All Three Files ───────────────────────────────────────────────────
print("\n── Downloading Files ─────────────────────────────────────────────────")
files.download('saved_models/doctor_appointment/da_biobart_weights.pt')
files.download('saved_models/doctor_appointment/da_qa_embeddings.npy')
files.download('saved_models/doctor_appointment/da_qa_data.csv')

print("\n✅ All 3 Doctor Appointment files downloaded.")
print("   → da_biobart_weights.pt")
print("   → da_qa_embeddings.npy")
print("   → da_qa_data.csv")



✅ Saved: da_biobart_weights.pt
✅ Saved: da_qa_embeddings.npy
✅ Saved: da_qa_data.csv

── Verification ──────────────────────────────────────────────────────
   da_biobart_weights.pt  : 666.1 MB
   da_qa_embeddings.npy   : shape (400, 384)
   da_qa_data.csv         : 400 rows, columns: ['question', 'answer', 'topic']

✅ Verification passed — embeddings and csv rows match perfectly

── Quick Inference Test ──────────────────────────────────────────────


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

   Query  : How do I book an appointment with a specialist at AIIMS?
   Answer : Provide the referral letter from the external specialist to the booking team and they will arrange an appropriate appointment within the relevant department....

✅ Inference test passed — model loads and generates correctly

── Downloading Files ─────────────────────────────────────────────────


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All 3 Doctor Appointment files downloaded.
   → da_biobart_weights.pt
   → da_qa_embeddings.npy
   → da_qa_data.csv


In [ ]:
train_emergency, val_emergency, test_emergency = prepare_model_dataset(df_emergency)


  Dataset Preparation
  Raw shape        : (400, 3)
  Clean shape      : (400, 5)
  Labels           : ['basic_first_aid', 'calling_emergency', 'disaster_evacuation', 'mental_health_emergency']

  Samples per label:
    basic_first_aid           :  129  █████████████████████████
    calling_emergency         :   90  ██████████████████
    disaster_evacuation       :   96  ███████████████████
    mental_health_emergency   :   85  █████████████████

  Question length  : mean=11.4 | max=18 | min=6
  Answer length    : mean=27.6 | max=39 | min=17

  Split Summary:
    Train :  280 rows  (70.0%)
    Val   :   60 rows  (15.0%)
    Test  :   60 rows  (15.0%)

  Stratification Check (label % per split):
    Label                     |  Train |    Val |   Test
    --------------------------------------------------
    basic_first_aid           |  32.1% |  31.7% |  33.3%
    calling_emergency         |  22.5% |  23.3% |  21.7%
    disaster_evacuation       |  23.9% |  23.3% |  25.0%
    mental_

In [ ]:
#emergency
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    BartForConditionalGeneration, BartTokenizer
)
from rouge_score import rouge_scorer
import numpy as np
import math
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Device ─────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Splits (already in memory) ────────────────────────────────────────────────
# train_emergency, val_emergency, test_emergency

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1 — FROZEN ENCODER + RETRIEVAL INDEX
# ══════════════════════════════════════════════════════════════════════════════

ENC_MODEL     = 'sentence-transformers/all-MiniLM-L6-v2'
enc_tokenizer = AutoTokenizer.from_pretrained(ENC_MODEL)
enc_model     = AutoModel.from_pretrained(ENC_MODEL).to(device)

for param in enc_model.parameters():
    param.requires_grad = False

print(f"Encoder        : {ENC_MODEL}")
print(f"Encoder params : "
      f"{sum(p.numel() for p in enc_model.parameters()):,}  (frozen)")

# ── Frozen Encoder Module ──────────────────────────────────────────────────────
class FrozenEncoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def mean_pool(self, output, attention_mask):
        tokens   = output.last_hidden_state
        mask_exp = attention_mask.unsqueeze(-1).expand(tokens.size()).float()
        return torch.sum(tokens * mask_exp, 1) / \
               torch.clamp(mask_exp.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            output = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )
        return self.mean_pool(output, attention_mask)

encoder = FrozenEncoder(enc_model).to(device)

# ── Encode Helper ──────────────────────────────────────────────────────────────
def encode_texts(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i : i + batch_size]
        encoded = enc_tokenizer(
            batch,
            max_length     = 128,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        embs = encoder(
            encoded['input_ids'].to(device),
            encoded['attention_mask'].to(device)
        )
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

# ── Build Retrieval Index from all splits ─────────────────────────────────────
print("\nBuilding retrieval index...")
full_df       = pd.concat(
    [train_emergency, val_emergency, test_emergency],
    ignore_index=True
).reset_index(drop=True)

full_qa_texts = (full_df['question'] + ' ' + full_df['answer']).tolist()
full_qa_embs  = encode_texts(full_qa_texts)

print(f"Indexed {len(full_df)} Q&A pairs across "
      f"{full_df['topic'].nunique()} topics")
for topic, count in full_df['topic'].value_counts().items():
    print(f"  {topic:25s} : {count} rows")

# ── Retrieval Function ─────────────────────────────────────────────────────────
def retrieve_top_k(query: str, k: int = 3) -> list:
    query_emb = encode_texts([query])
    q_norm    = nn.functional.normalize(
                    torch.tensor(query_emb,     dtype=torch.float32), dim=-1)
    k_norm    = nn.functional.normalize(
                    torch.tensor(full_qa_embs,  dtype=torch.float32), dim=-1)
    sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
    top_k_idx = torch.topk(sims, k=k).indices.tolist()

    return [{
        'question'   : full_df.iloc[i]['question'],
        'answer'     : full_df.iloc[i]['answer'],
        'topic'      : full_df.iloc[i]['topic'],
        'similarity' : round(sims[i].item(), 4)
    } for i in top_k_idx]

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2 — BIOBART COMPOSER
# ══════════════════════════════════════════════════════════════════════════════

BART_MODEL     = 'GanjinZero/biobart-v2-base'
bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL)
bart_model     = BartForConditionalGeneration.from_pretrained(
                     BART_MODEL).to(device)

print(f"\nComposer       : {BART_MODEL}")
print(f"Composer params: "
      f"{sum(p.numel() for p in bart_model.parameters()):,}")

# ── Dataset ────────────────────────────────────────────────────────────────────
class EmergencyRAGDataset(Dataset):
    def __init__(self, df, retrieve_fn, k=3):
        self.data    = df.reset_index(drop=True)
        self.samples = self._build_samples(retrieve_fn, k)

    def _build_samples(self, retrieve_fn, k):
        samples = []
        for idx in range(len(self.data)):
            row         = self.data.loc[idx]
            context     = retrieve_fn(row['question'], k=k)
            context_str = ' '.join([
                f"Context {i+1}: {c['answer']}"
                for i, c in enumerate(context)
            ])
            samples.append({
                'input'    : (f"Question: {row['question']} "
                              f"{context_str} "
                              f"Compose a precise and accurate "
                              f"emergency medical answer:"),
                'target'   : row['answer'],
                'topic'    : row['topic'],
                'question' : row['question']
            })
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# ── Collator ───────────────────────────────────────────────────────────────────
class RAGCollator:
    def __init__(self, tokenizer, max_input_len=1024, max_target_len=256):
        self.tokenizer      = tokenizer
        self.max_input_len  = max_input_len
        self.max_target_len = max_target_len

    def __call__(self, batch):
        inputs    = [b['input']    for b in batch]
        targets   = [b['target']   for b in batch]
        topics    = [b['topic']    for b in batch]
        questions = [b['question'] for b in batch]

        inp_enc = self.tokenizer(
            inputs,
            max_length     = self.max_input_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        tgt_enc = self.tokenizer(
            targets,
            max_length     = self.max_target_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )

        labels = tgt_enc['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids'      : inp_enc['input_ids'],
            'attention_mask' : inp_enc['attention_mask'],
            'labels'         : labels,
            'topics'         : topics,
            'questions'      : questions,
            'raw_targets'    : targets
        }

collator = RAGCollator(bart_tokenizer)

# ── DataLoaders ────────────────────────────────────────────────────────────────
print("\nBuilding RAG datasets...")
train_dataset = EmergencyRAGDataset(train_emergency, retrieve_top_k, k=3)
val_dataset   = EmergencyRAGDataset(val_emergency,   retrieve_top_k, k=3)
test_dataset  = EmergencyRAGDataset(test_emergency,  retrieve_top_k, k=3)

train_loader  = DataLoader(train_dataset, batch_size=4,
                            shuffle=True,  collate_fn=collator)
val_loader    = DataLoader(val_dataset,   batch_size=4,
                            shuffle=False, collate_fn=collator)
test_loader   = DataLoader(test_dataset,  batch_size=4,
                            shuffle=False, collate_fn=collator)

print(f"Train : {len(train_dataset)} samples | {len(train_loader)} batches")
print(f"Val   : {len(val_dataset)}   samples | {len(val_loader)}   batches")
print(f"Test  : {len(test_dataset)}  samples | {len(test_loader)}  batches")

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 3 — TRAINING ENGINE
# ══════════════════════════════════════════════════════════════════════════════

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer    = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.min_lr       = min_lr
        self.current_step = 0
        self.base_lrs     = [pg['lr'] for pg in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        for pg, lr in zip(self.optimizer.param_groups, self._compute_lr()):
            pg['lr'] = lr

    def _compute_lr(self):
        s = self.current_step
        lrs = []
        for base_lr in self.base_lrs:
            if s < self.warmup_steps:
                lr = base_lr * (s / max(1, self.warmup_steps))
            else:
                progress = (s - self.warmup_steps) / \
                           max(1, self.total_steps - self.warmup_steps)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * \
                     (1 + math.cos(math.pi * progress))
            lrs.append(lr)
        return lrs

class EmergencyRAGTrainer:
    def __init__(self, model, optimizer, scheduler, tokenizer,
                 device, accum_steps=4, max_grad_norm=1.0,
                 checkpoint_path='best_emergency_rag.pt'):
        self.model           = model
        self.optimizer       = optimizer
        self.scheduler       = scheduler
        self.tokenizer       = tokenizer
        self.device          = device
        self.accum_steps     = accum_steps
        self.max_grad_norm   = max_grad_norm
        self.checkpoint_path = checkpoint_path
        self.best_val_loss   = float('inf')
        self.patience_count  = 0

    def train_epoch(self, loader):
        self.model.train()
        total_loss, total_tokens = 0.0, 0
        self.optimizer.zero_grad()

        for step, batch in enumerate(loader):
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels         = batch['labels'].to(self.device)

            outputs        = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels
            )
            loss           = outputs.loss / self.accum_steps
            loss.backward()

            n_tokens       = (labels != -100).sum().item()
            total_loss    += outputs.loss.item() * n_tokens
            total_tokens  += n_tokens

            if (step + 1) % self.accum_steps == 0 or \
               (step + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.max_grad_norm
                )
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()

        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity

    def evaluate(self, loader):
        self.model.eval()
        total_loss, total_tokens  = 0.0, 0
        all_preds, all_refs       = [], []
        all_topics, all_questions = [], []

        with torch.no_grad():
            for batch in loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['labels'].to(self.device)

                outputs        = self.model(
                    input_ids      = input_ids,
                    attention_mask = attention_mask,
                    labels         = labels
                )
                n_tokens       = (labels != -100).sum().item()
                total_loss    += outputs.loss.item() * n_tokens
                total_tokens  += n_tokens

                generated = self.model.generate(
                    input_ids            = input_ids,
                    attention_mask       = attention_mask,
                    max_new_tokens       = 256,
                    num_beams            = 4,
                    early_stopping       = True,
                    no_repeat_ngram_size = 3,
                    length_penalty       = 1.0
                )

                preds     = self.tokenizer.batch_decode(
                    generated, skip_special_tokens=True
                )
                label_ids = labels.clone()
                label_ids[label_ids == -100] = self.tokenizer.pad_token_id
                refs      = self.tokenizer.batch_decode(
                    label_ids, skip_special_tokens=True
                )

                all_preds.extend(preds)
                all_refs.extend(refs)
                all_topics.extend(batch['topics'])
                all_questions.extend(batch['questions'])

        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity, all_preds, all_refs, \
               all_topics, all_questions

    def check_early_stop(self, val_loss, patience=3):
        if val_loss < self.best_val_loss:
            self.best_val_loss  = val_loss
            self.patience_count = 0
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : val_loss
            }, self.checkpoint_path)
            return False
        self.patience_count += 1
        return self.patience_count >= patience

    def load_best(self):
        if not os.path.exists(self.checkpoint_path):
            print("⚠️  No checkpoint found — saving current state...")
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : self.best_val_loss
            }, self.checkpoint_path)
        ckpt = torch.load(self.checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt['model_state'])
        print(f"✅ Best checkpoint restored — "
              f"Val Loss: {ckpt['val_loss']:.4f}")

# ── Metrics ────────────────────────────────────────────────────────────────────
rouge_scorer_obj = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
)

def compute_rouge(preds, refs):
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = rouge_scorer_obj.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    return {
        'rouge1' : round(np.mean(r1), 4),
        'rouge2' : round(np.mean(r2), 4),
        'rougeL' : round(np.mean(rL), 4)
    }

def compute_token_f1(pred, ref):
    pred_tokens = pred.lower().split()
    ref_tokens  = ref.lower().split()
    common      = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    p = len(common) / len(pred_tokens) if pred_tokens else 0
    r = len(common) / len(ref_tokens)  if ref_tokens  else 0
    return round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0.0

def print_metrics(split_name, loss, ppl, preds, refs, topics, questions):
    rouge     = compute_rouge(preds, refs)
    token_f1s = [compute_token_f1(p, r) for p, r in zip(preds, refs)]
    em        = sum(p.strip().lower() == r.strip().lower()
                    for p, r in zip(preds, refs)) / len(preds)

    print(f"\n{'='*60}")
    print(f"  Metrics — {split_name}")
    print(f"{'='*60}")
    print(f"  Loss        : {loss:.4f}")
    print(f"  Perplexity  : {ppl:.4f}")
    print(f"  ROUGE-1     : {rouge['rouge1']:.4f}")
    print(f"  ROUGE-2     : {rouge['rouge2']:.4f}")
    print(f"  ROUGE-L     : {rouge['rougeL']:.4f}")
    print(f"  Token F1    : {np.mean(token_f1s):.4f}")
    print(f"  Exact Match : {em:.4f}")
    print(f"\n  Per-Topic ROUGE-L:")
    for topic in sorted(set(topics)):
        mask    = [t == topic for t in topics]
        t_preds = [p for p, m in zip(preds, mask) if m]
        t_refs  = [r for r, m in zip(refs,  mask) if m]
        t_rouge = compute_rouge(t_preds, t_refs)
        t_f1    = np.mean([compute_token_f1(p, r)
                           for p, r in zip(t_preds, t_refs)])
        print(f"    {topic:25s} : ROUGE-L {t_rouge['rougeL']:.4f} | "
              f"Token F1 {t_f1:.4f} | n={len(t_preds)}")

# ── Initialise ─────────────────────────────────────────────────────────────────
EPOCHS       = 15
ACCUM_STEPS  = 4
total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = total_steps // 10

optimizer = torch.optim.AdamW(
    bart_model.parameters(), lr=3e-5, weight_decay=0.01
)
scheduler = WarmupCosineScheduler(optimizer, warmup_steps, total_steps)
trainer   = EmergencyRAGTrainer(
    bart_model, optimizer, scheduler,
    bart_tokenizer, device,
    accum_steps     = ACCUM_STEPS,
    checkpoint_path = 'best_emergency_rag.pt'
)

# ── Training Loop ──────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  Training Emergency BioBART RAG  |  {EPOCHS} epochs  |  "
      f"Warmup {warmup_steps}  |  Total {total_steps} steps")
print(f"{'='*70}")
print(f"  {'Epoch':>5} | {'Train Loss':>10} | {'Train PPL':>9} | "
      f"{'Val Loss':>8} | {'Val PPL':>7} | {'ROUGE-L':>7} | Status")
print(f"  {'-'*70}")

for epoch in range(EPOCHS):
    tr_loss, tr_ppl       = trainer.train_epoch(train_loader)
    vl_loss, vl_ppl, \
    vl_preds, vl_refs, \
    vl_topics, vl_qs      = trainer.evaluate(val_loader)
    vl_rouge              = compute_rouge(vl_preds, vl_refs)
    stop                  = trainer.check_early_stop(vl_loss, patience=3)
    status                = "← best ✅" if trainer.patience_count == 0 \
                            else f"patience {trainer.patience_count}/3"

    print(f"  {epoch+1:>5} | {tr_loss:>10.4f} | {tr_ppl:>9.4f} | "
          f"{vl_loss:>8.4f} | {vl_ppl:>7.4f} | "
          f"{vl_rouge['rougeL']:>7.4f} | {status}")

    if stop:
        print(f"\n  Early stopping at epoch {epoch+1}")
        break

# ══════════════════════════════════════════════════════════════════════════════
#  METRICS — ALL SPLITS + SAMPLE PREDICTIONS
# ══════════════════════════════════════════════════════════════════════════════

trainer.load_best()

tr_loss, tr_ppl, tr_preds, \
tr_refs, tr_topics, tr_qs  = trainer.evaluate(train_loader)

vl_loss, vl_ppl, vl_preds, \
vl_refs, vl_topics, vl_qs  = trainer.evaluate(val_loader)

ts_loss, ts_ppl, ts_preds, \
ts_refs, ts_topics, ts_qs  = trainer.evaluate(test_loader)

print_metrics("Train",      tr_loss, tr_ppl,
              tr_preds, tr_refs, tr_topics, tr_qs)
print_metrics("Validation", vl_loss, vl_ppl,
              vl_preds, vl_refs, vl_topics, vl_qs)
print_metrics("Test",       ts_loss, ts_ppl,
              ts_preds, ts_refs, ts_topics, ts_qs)

# ── Sample Predictions from Test Set ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Sample Predictions — Test Set (5 random)")
print(f"{'='*60}")

for i in np.random.choice(len(ts_preds), 5, replace=False):
    r = compute_rouge([ts_preds[i]], [ts_refs[i]])
    f = compute_token_f1(ts_preds[i], ts_refs[i])
    print(f"\n  Topic     : {ts_topics[i]}")
    print(f"  Question  : {ts_qs[i]}")
    print(f"  Reference : {ts_refs[i][:150]}...")
    print(f"  Generated : {ts_preds[i][:150]}...")
    print(f"  ROUGE-L   : {r['rougeL']:.4f} | Token F1 : {f:.4f}")
    print(f"  {'-'*55}")

# ══════════════════════════════════════════════════════════════════════════════
#  STANDALONE TEST QUERIES — one per topic
# ══════════════════════════════════════════════════════════════════════════════

def answer_emergency_query(question: str, k: int = 3) -> dict:
    retrieved   = retrieve_top_k(question, k=k)
    context_str = ' '.join([
        f"Context {i+1}: {r['answer']}"
        for i, r in enumerate(retrieved)
    ])
    input_text  = (
        f"Question: {question} "
        f"{context_str} "
        f"Compose a precise and accurate emergency medical answer:"
    )

    bart_model.eval()
    inputs = bart_tokenizer(
        input_text,
        max_length     = 1024,
        truncation     = True,
        return_tensors = 'pt'
    ).to(device)

    with torch.no_grad():
        output = bart_model.generate(
            input_ids            = inputs['input_ids'],
            attention_mask       = inputs['attention_mask'],
            max_new_tokens       = 256,
            num_beams            = 4,
            early_stopping       = True,
            no_repeat_ngram_size = 3,
            length_penalty       = 1.0
        )

    answer = bart_tokenizer.decode(output[0], skip_special_tokens=True)
    return {
        'question'  : question,
        'answer'    : answer,
        'retrieved' : [{
            'topic'      : r['topic'],
            'question'   : r['question'],
            'similarity' : r['similarity']
        } for r in retrieved]
    }

test_queries = [
    # calling_emergency
    "What number should I call for an ambulance in India?",
    # basic_first_aid
    "What should I do if someone is choking and cannot breathe?",
    # mental_health_emergency
    "Someone I know is threatening to hurt themselves — what should I do?",
    # disaster_evacuation
    "What should I do if a fire alarm goes off in the hospital?"
]

print(f"\n{'='*60}")
print(f"  Standalone Test Queries — Emergency Model")
print(f"{'='*60}")

for q in test_queries:
    out = answer_emergency_query(q)
    print(f"\n  Q         : {out['question']}")
    print(f"  A         : {out['answer']}")
    print(f"  Retrieved :")
    for r in out['retrieved']:
        print(f"    [{r['similarity']}] "
              f"{r['topic']:25s} — {r['question'][:50]}...")
    print(f"  {'-'*55}")

Device : cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder        : sentence-transformers/all-MiniLM-L6-v2
Encoder params : 22,713,216  (frozen)

Building retrieval index...
Indexed 400 Q&A pairs across 4 topics
  basic_first_aid           : 129 rows
  disaster_evacuation       : 96 rows
  calling_emergency         : 90 rows
  mental_health_emergency   : 85 rows


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


Composer       : GanjinZero/biobart-v2-base
Composer params: 166,404,864

Building RAG datasets...
Train : 280 samples | 70 batches
Val   : 60   samples | 15   batches
Test  : 60  samples | 15  batches

  Training Emergency BioBART RAG  |  15 epochs  |  Warmup 25  |  Total 255 steps
  Epoch | Train Loss | Train PPL | Val Loss | Val PPL | ROUGE-L | Status
  ----------------------------------------------------------------------
      1 |     0.6848 |    1.9834 |   0.0430 |  1.0439 |  0.9807 | ← best ✅
      2 |     0.0502 |    1.0515 |   0.0047 |  1.0047 |  0.9954 | ← best ✅
      3 |     0.0124 |    1.0124 |   0.0038 |  1.0038 |  0.9961 | ← best ✅
      4 |     0.0046 |    1.0046 |   0.0033 |  1.0033 |  0.9940 | ← best ✅
      5 |     0.0160 |    1.0161 |   0.0035 |  1.0036 |  0.9931 | patience 1/3
      6 |     0.0018 |    1.0018 |   0.0038 |  1.0038 |  0.9950 | patience 2/3
      7 |     0.0008 |    1.0008 |   0.0039 |  1.0039 |  0.9948 | patience 3/3

  Early stopping at epoch 7
✅ B

In [ ]:
# ── Save Emergency Model Files ────────────────────────────────────────────────
import os
import numpy as np
from google.colab import files

os.makedirs('saved_models/emergency', exist_ok=True)

# ── Save 1: Trained BioBART Weights ───────────────────────────────────────────
torch.save(
    bart_model.state_dict(),
    'saved_models/emergency/emergency_biobart_weights.pt'
)
print("✅ Saved: emergency_biobart_weights.pt")

# ── Save 2: Retrieval Embeddings ───────────────────────────────────────────────
np.save(
    'saved_models/emergency/emergency_qa_embeddings.npy',
    full_qa_embs
)
print("✅ Saved: emergency_qa_embeddings.npy")

# ── Save 3: Q&A Data ───────────────────────────────────────────────────────────
full_df.to_csv(
    'saved_models/emergency/emergency_qa_data.csv',
    index=False
)
print("✅ Saved: emergency_qa_data.csv")

# ── Verify Everything Before Downloading ──────────────────────────────────────
print("\n── Verification ──────────────────────────────────────────────────────")

# Check 1: weights file size
weights_size = os.path.getsize(
    'saved_models/emergency/emergency_biobart_weights.pt'
)
print(f"   emergency_biobart_weights.pt  : {weights_size / 1e6:.1f} MB")

# Check 2: embeddings shape
test_emb_load = np.load(
    'saved_models/emergency/emergency_qa_embeddings.npy'
)
print(f"   emergency_qa_embeddings.npy   : shape {test_emb_load.shape}")

# Check 3: csv rows
test_csv_load = pd.read_csv(
    'saved_models/emergency/emergency_qa_data.csv'
)
print(f"   emergency_qa_data.csv         : {len(test_csv_load)} rows, "
      f"columns: {list(test_csv_load.columns)}")

# Check 4: embeddings rows must match csv rows
assert test_emb_load.shape[0] == len(test_csv_load), \
    f"❌ MISMATCH — embeddings {test_emb_load.shape[0]} rows " \
    f"vs csv {len(test_csv_load)} rows"
print(f"\n✅ Verification passed — embeddings and csv rows match perfectly")

# ── Quick End-to-End Inference Test Before Downloading ────────────────────────
print("\n── Quick Inference Test ──────────────────────────────────────────────")

import torch.nn.functional as F
from transformers import BartForConditionalGeneration, BartTokenizer

# Reload weights into fresh model to simulate PyCharm loading
test_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
test_model.load_state_dict(
    torch.load(
        'saved_models/emergency/emergency_biobart_weights.pt',
        map_location=device
    )
)
test_model.to(device)
test_model.eval()

# Run one test query through the full pipeline
test_query     = "What should I do if someone is having a heart attack?"
test_query_emb = encode_texts([test_query])

# Retrieve top 3
q_norm    = F.normalize(
    torch.tensor(test_query_emb, dtype=torch.float32), dim=-1
)
k_norm    = F.normalize(
    torch.tensor(test_emb_load,  dtype=torch.float32), dim=-1
)
sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
top_k_idx = torch.topk(sims, k=3).indices.tolist()

context_str = ' '.join([
    f"Context {i+1}: {test_csv_load.iloc[idx]['answer']}"
    for i, idx in enumerate(top_k_idx)
])

input_text = (
    f"Question: {test_query} "
    f"{context_str} "
    f"Compose a precise and accurate emergency medical answer:"
)

inputs = bart_tokenizer(
    input_text,
    max_length     = 1024,
    truncation     = True,
    return_tensors = 'pt'
).to(device)

with torch.no_grad():
    output = test_model.generate(
        input_ids            = inputs['input_ids'],
        attention_mask       = inputs['attention_mask'],
        max_new_tokens       = 256,
        num_beams            = 4,
        early_stopping       = True,
        no_repeat_ngram_size = 3,
        length_penalty       = 1.0
    )

test_answer = bart_tokenizer.decode(output[0], skip_special_tokens=True)
print(f"   Query  : {test_query}")
print(f"   Answer : {test_answer[:200]}...")
print(f"\n✅ Inference test passed — model loads and generates correctly")

# ── Download All Three Files ───────────────────────────────────────────────────
print("\n── Downloading Files ─────────────────────────────────────────────────")
files.download('saved_models/emergency/emergency_biobart_weights.pt')
files.download('saved_models/emergency/emergency_qa_embeddings.npy')
files.download('saved_models/emergency/emergency_qa_data.csv')

print("\n✅ All 3 Emergency files downloaded.")
print("   → emergency_biobart_weights.pt")
print("   → emergency_qa_embeddings.npy")
print("   → emergency_qa_data.csv")


✅ Saved: emergency_biobart_weights.pt
✅ Saved: emergency_qa_embeddings.npy
✅ Saved: emergency_qa_data.csv

── Verification ──────────────────────────────────────────────────────
   emergency_biobart_weights.pt  : 666.1 MB
   emergency_qa_embeddings.npy   : shape (400, 384)
   emergency_qa_data.csv         : 400 rows, columns: ['question', 'answer', 'topic']

✅ Verification passed — embeddings and csv rows match perfectly

── Quick Inference Test ──────────────────────────────────────────────


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

   Query  : What should I do if someone is having a heart attack?
   Answer : Call 112 immediately, ask the person to sit or lie in a comfortable position, loosen tight clothing, give one 325mg aspirin if available and that person is not allergic, and stay with them....

✅ Inference test passed — model loads and generates correctly

── Downloading Files ─────────────────────────────────────────────────


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All 3 Emergency files downloaded.
   → emergency_biobart_weights.pt
   → emergency_qa_embeddings.npy
   → emergency_qa_data.csv


In [ ]:
train_pharma, val_pharma, test_pharma = prepare_model_dataset(df_pharma)


  Dataset Preparation
  Raw shape        : (400, 3)
  Clean shape      : (400, 5)
  Labels           : ['general_queries', 'medicine_costs', 'otc_medicines', 'prescription', 'side_effects']

  Samples per label:
    general_queries           :  106  █████████████████████
    medicine_costs            :   65  █████████████
    otc_medicines             :   76  ███████████████
    prescription              :   75  ███████████████
    side_effects              :   78  ███████████████

  Question length  : mean=11.4 | max=20 | min=5
  Answer length    : mean=27.4 | max=39 | min=17

  Split Summary:
    Train :  280 rows  (70.0%)
    Val   :   60 rows  (15.0%)
    Test  :   60 rows  (15.0%)

  Stratification Check (label % per split):
    Label                     |  Train |    Val |   Test
    --------------------------------------------------
    general_queries           |  26.4% |  26.7% |  26.7%
    medicine_costs            |  16.1% |  16.7% |  16.7%
    otc_medicines             |  

In [ ]:
#pharma
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    BartForConditionalGeneration, BartTokenizer
)
from rouge_score import rouge_scorer
import numpy as np
import math
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Device ─────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ── Splits (already in memory) ────────────────────────────────────────────────
# train_pharma, val_pharma, test_pharma

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1 — FROZEN ENCODER + RETRIEVAL INDEX
# ══════════════════════════════════════════════════════════════════════════════

ENC_MODEL     = 'sentence-transformers/all-MiniLM-L6-v2'
enc_tokenizer = AutoTokenizer.from_pretrained(ENC_MODEL)
enc_model     = AutoModel.from_pretrained(ENC_MODEL).to(device)

for param in enc_model.parameters():
    param.requires_grad = False

print(f"Encoder        : {ENC_MODEL}")
print(f"Encoder params : "
      f"{sum(p.numel() for p in enc_model.parameters()):,}  (frozen)")

# ── Frozen Encoder Module ──────────────────────────────────────────────────────
class FrozenEncoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def mean_pool(self, output, attention_mask):
        tokens   = output.last_hidden_state
        mask_exp = attention_mask.unsqueeze(-1).expand(tokens.size()).float()
        return torch.sum(tokens * mask_exp, 1) / \
               torch.clamp(mask_exp.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            output = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )
        return self.mean_pool(output, attention_mask)

encoder = FrozenEncoder(enc_model).to(device)

# ── Encode Helper ──────────────────────────────────────────────────────────────
def encode_texts(texts, batch_size=32):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch   = texts[i : i + batch_size]
        encoded = enc_tokenizer(
            batch,
            max_length     = 128,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        embs = encoder(
            encoded['input_ids'].to(device),
            encoded['attention_mask'].to(device)
        )
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

# ── Build Retrieval Index from all splits ─────────────────────────────────────
print("\nBuilding retrieval index...")
full_df       = pd.concat(
    [train_pharma, val_pharma, test_pharma],
    ignore_index=True
).reset_index(drop=True)

full_qa_texts = (full_df['question'] + ' ' + full_df['answer']).tolist()
full_qa_embs  = encode_texts(full_qa_texts)

print(f"Indexed {len(full_df)} Q&A pairs across "
      f"{full_df['topic'].nunique()} topics")
for topic, count in full_df['topic'].value_counts().items():
    print(f"  {topic:25s} : {count} rows")

# ── Retrieval Function ─────────────────────────────────────────────────────────
def retrieve_top_k(query: str, k: int = 3) -> list:
    query_emb = encode_texts([query])
    q_norm    = nn.functional.normalize(
                    torch.tensor(query_emb,     dtype=torch.float32), dim=-1)
    k_norm    = nn.functional.normalize(
                    torch.tensor(full_qa_embs,  dtype=torch.float32), dim=-1)
    sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
    top_k_idx = torch.topk(sims, k=k).indices.tolist()

    return [{
        'question'   : full_df.iloc[i]['question'],
        'answer'     : full_df.iloc[i]['answer'],
        'topic'      : full_df.iloc[i]['topic'],
        'similarity' : round(sims[i].item(), 4)
    } for i in top_k_idx]

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2 — BIOBART COMPOSER
# ══════════════════════════════════════════════════════════════════════════════

BART_MODEL     = 'GanjinZero/biobart-v2-base'
bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL)
bart_model     = BartForConditionalGeneration.from_pretrained(
                     BART_MODEL).to(device)

print(f"\nComposer       : {BART_MODEL}")
print(f"Composer params: "
      f"{sum(p.numel() for p in bart_model.parameters()):,}")

# ── Dataset ────────────────────────────────────────────────────────────────────
class PharmacyRAGDataset(Dataset):
    def __init__(self, df, retrieve_fn, k=3):
        self.data    = df.reset_index(drop=True)
        self.samples = self._build_samples(retrieve_fn, k)

    def _build_samples(self, retrieve_fn, k):
        samples = []
        for idx in range(len(self.data)):
            row         = self.data.loc[idx]
            context     = retrieve_fn(row['question'], k=k)
            context_str = ' '.join([
                f"Context {i+1}: {c['answer']}"
                for i, c in enumerate(context)
            ])
            samples.append({
                'input'    : (f"Question: {row['question']} "
                              f"{context_str} "
                              f"Compose a precise and accurate "
                              f"hospital pharmacy answer:"),
                'target'   : row['answer'],
                'topic'    : row['topic'],
                'question' : row['question']
            })
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# ── Collator ───────────────────────────────────────────────────────────────────
class RAGCollator:
    def __init__(self, tokenizer, max_input_len=1024, max_target_len=256):
        self.tokenizer      = tokenizer
        self.max_input_len  = max_input_len
        self.max_target_len = max_target_len

    def __call__(self, batch):
        inputs    = [b['input']    for b in batch]
        targets   = [b['target']   for b in batch]
        topics    = [b['topic']    for b in batch]
        questions = [b['question'] for b in batch]

        inp_enc = self.tokenizer(
            inputs,
            max_length     = self.max_input_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )
        tgt_enc = self.tokenizer(
            targets,
            max_length     = self.max_target_len,
            padding        = 'longest',
            truncation     = True,
            return_tensors = 'pt'
        )

        labels = tgt_enc['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids'      : inp_enc['input_ids'],
            'attention_mask' : inp_enc['attention_mask'],
            'labels'         : labels,
            'topics'         : topics,
            'questions'      : questions,
            'raw_targets'    : targets
        }

collator = RAGCollator(bart_tokenizer)

# ── DataLoaders ────────────────────────────────────────────────────────────────
print("\nBuilding RAG datasets...")
train_dataset = PharmacyRAGDataset(train_pharma, retrieve_top_k, k=3)
val_dataset   = PharmacyRAGDataset(val_pharma,   retrieve_top_k, k=3)
test_dataset  = PharmacyRAGDataset(test_pharma,  retrieve_top_k, k=3)

train_loader  = DataLoader(train_dataset, batch_size=4,
                            shuffle=True,  collate_fn=collator)
val_loader    = DataLoader(val_dataset,   batch_size=4,
                            shuffle=False, collate_fn=collator)
test_loader   = DataLoader(test_dataset,  batch_size=4,
                            shuffle=False, collate_fn=collator)

print(f"Train : {len(train_dataset)} samples | {len(train_loader)} batches")
print(f"Val   : {len(val_dataset)}   samples | {len(val_loader)}   batches")
print(f"Test  : {len(test_dataset)}  samples | {len(test_loader)}  batches")

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 3 — TRAINING ENGINE
# ══════════════════════════════════════════════════════════════════════════════

class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer    = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.min_lr       = min_lr
        self.current_step = 0
        self.base_lrs     = [pg['lr'] for pg in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        for pg, lr in zip(self.optimizer.param_groups, self._compute_lr()):
            pg['lr'] = lr

    def _compute_lr(self):
        s = self.current_step
        lrs = []
        for base_lr in self.base_lrs:
            if s < self.warmup_steps:
                lr = base_lr * (s / max(1, self.warmup_steps))
            else:
                progress = (s - self.warmup_steps) / \
                           max(1, self.total_steps - self.warmup_steps)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * \
                     (1 + math.cos(math.pi * progress))
            lrs.append(lr)
        return lrs

class PharmacyRAGTrainer:
    def __init__(self, model, optimizer, scheduler, tokenizer,
                 device, accum_steps=4, max_grad_norm=1.0,
                 checkpoint_path='best_pharma_rag.pt'):
        self.model           = model
        self.optimizer       = optimizer
        self.scheduler       = scheduler
        self.tokenizer       = tokenizer
        self.device          = device
        self.accum_steps     = accum_steps
        self.max_grad_norm   = max_grad_norm
        self.checkpoint_path = checkpoint_path
        self.best_val_loss   = float('inf')
        self.patience_count  = 0

    def train_epoch(self, loader):
        self.model.train()
        total_loss, total_tokens = 0.0, 0
        self.optimizer.zero_grad()

        for step, batch in enumerate(loader):
            input_ids      = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels         = batch['labels'].to(self.device)

            outputs        = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels
            )
            loss           = outputs.loss / self.accum_steps
            loss.backward()

            n_tokens       = (labels != -100).sum().item()
            total_loss    += outputs.loss.item() * n_tokens
            total_tokens  += n_tokens

            if (step + 1) % self.accum_steps == 0 or \
               (step + 1) == len(loader):
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.max_grad_norm
                )
                self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()

        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity

    def evaluate(self, loader):
        self.model.eval()
        total_loss, total_tokens  = 0.0, 0
        all_preds, all_refs       = [], []
        all_topics, all_questions = [], []

        with torch.no_grad():
            for batch in loader:
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['labels'].to(self.device)

                outputs        = self.model(
                    input_ids      = input_ids,
                    attention_mask = attention_mask,
                    labels         = labels
                )
                n_tokens       = (labels != -100).sum().item()
                total_loss    += outputs.loss.item() * n_tokens
                total_tokens  += n_tokens

                generated = self.model.generate(
                    input_ids            = input_ids,
                    attention_mask       = attention_mask,
                    max_new_tokens       = 256,
                    num_beams            = 4,
                    early_stopping       = True,
                    no_repeat_ngram_size = 3,
                    length_penalty       = 1.0
                )

                preds     = self.tokenizer.batch_decode(
                    generated, skip_special_tokens=True
                )
                label_ids = labels.clone()
                label_ids[label_ids == -100] = self.tokenizer.pad_token_id
                refs      = self.tokenizer.batch_decode(
                    label_ids, skip_special_tokens=True
                )

                all_preds.extend(preds)
                all_refs.extend(refs)
                all_topics.extend(batch['topics'])
                all_questions.extend(batch['questions'])

        avg_loss   = total_loss / max(total_tokens, 1)
        perplexity = math.exp(min(avg_loss, 100))
        return avg_loss, perplexity, all_preds, all_refs, \
               all_topics, all_questions

    def check_early_stop(self, val_loss, patience=3):
        if val_loss < self.best_val_loss:
            self.best_val_loss  = val_loss
            self.patience_count = 0
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : val_loss
            }, self.checkpoint_path)
            return False
        self.patience_count += 1
        return self.patience_count >= patience

    def load_best(self):
        if not os.path.exists(self.checkpoint_path):
            print("⚠️  No checkpoint found — saving current state...")
            torch.save({
                'model_state' : self.model.state_dict(),
                'val_loss'    : self.best_val_loss
            }, self.checkpoint_path)
        ckpt = torch.load(self.checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt['model_state'])
        print(f"✅ Best checkpoint restored — "
              f"Val Loss: {ckpt['val_loss']:.4f}")

# ── Metrics ────────────────────────────────────────────────────────────────────
rouge_scorer_obj = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
)

def compute_rouge(preds, refs):
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = rouge_scorer_obj.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    return {
        'rouge1' : round(np.mean(r1), 4),
        'rouge2' : round(np.mean(r2), 4),
        'rougeL' : round(np.mean(rL), 4)
    }

def compute_token_f1(pred, ref):
    pred_tokens = pred.lower().split()
    ref_tokens  = ref.lower().split()
    common      = set(pred_tokens) & set(ref_tokens)
    if not common:
        return 0.0
    p = len(common) / len(pred_tokens) if pred_tokens else 0
    r = len(common) / len(ref_tokens)  if ref_tokens  else 0
    return round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0.0

def print_metrics(split_name, loss, ppl, preds, refs, topics, questions):
    rouge     = compute_rouge(preds, refs)
    token_f1s = [compute_token_f1(p, r) for p, r in zip(preds, refs)]
    em        = sum(p.strip().lower() == r.strip().lower()
                    for p, r in zip(preds, refs)) / len(preds)

    print(f"\n{'='*60}")
    print(f"  Metrics — {split_name}")
    print(f"{'='*60}")
    print(f"  Loss        : {loss:.4f}")
    print(f"  Perplexity  : {ppl:.4f}")
    print(f"  ROUGE-1     : {rouge['rouge1']:.4f}")
    print(f"  ROUGE-2     : {rouge['rouge2']:.4f}")
    print(f"  ROUGE-L     : {rouge['rougeL']:.4f}")
    print(f"  Token F1    : {np.mean(token_f1s):.4f}")
    print(f"  Exact Match : {em:.4f}")
    print(f"\n  Per-Topic ROUGE-L:")
    for topic in sorted(set(topics)):
        mask    = [t == topic for t in topics]
        t_preds = [p for p, m in zip(preds, mask) if m]
        t_refs  = [r for r, m in zip(refs,  mask) if m]
        t_rouge = compute_rouge(t_preds, t_refs)
        t_f1    = np.mean([compute_token_f1(p, r)
                           for p, r in zip(t_preds, t_refs)])
        print(f"    {topic:25s} : ROUGE-L {t_rouge['rougeL']:.4f} | "
              f"Token F1 {t_f1:.4f} | n={len(t_preds)}")

# ── Initialise ─────────────────────────────────────────────────────────────────
EPOCHS       = 15
ACCUM_STEPS  = 4
total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = total_steps // 10

optimizer = torch.optim.AdamW(
    bart_model.parameters(), lr=3e-5, weight_decay=0.01
)
scheduler = WarmupCosineScheduler(optimizer, warmup_steps, total_steps)
trainer   = PharmacyRAGTrainer(
    bart_model, optimizer, scheduler,
    bart_tokenizer, device,
    accum_steps     = ACCUM_STEPS,
    checkpoint_path = 'best_pharma_rag.pt'
)

# ── Training Loop ──────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  Training Pharmacy BioBART RAG  |  {EPOCHS} epochs  |  "
      f"Warmup {warmup_steps}  |  Total {total_steps} steps")
print(f"{'='*70}")
print(f"  {'Epoch':>5} | {'Train Loss':>10} | {'Train PPL':>9} | "
      f"{'Val Loss':>8} | {'Val PPL':>7} | {'ROUGE-L':>7} | Status")
print(f"  {'-'*70}")

for epoch in range(EPOCHS):
    tr_loss, tr_ppl       = trainer.train_epoch(train_loader)
    vl_loss, vl_ppl, \
    vl_preds, vl_refs, \
    vl_topics, vl_qs      = trainer.evaluate(val_loader)
    vl_rouge              = compute_rouge(vl_preds, vl_refs)
    stop                  = trainer.check_early_stop(vl_loss, patience=3)
    status                = "← best ✅" if trainer.patience_count == 0 \
                            else f"patience {trainer.patience_count}/3"

    print(f"  {epoch+1:>5} | {tr_loss:>10.4f} | {tr_ppl:>9.4f} | "
          f"{vl_loss:>8.4f} | {vl_ppl:>7.4f} | "
          f"{vl_rouge['rougeL']:>7.4f} | {status}")

    if stop:
        print(f"\n  Early stopping at epoch {epoch+1}")
        break

# ══════════════════════════════════════════════════════════════════════════════
#  METRICS — ALL SPLITS + SAMPLE PREDICTIONS
# ══════════════════════════════════════════════════════════════════════════════

trainer.load_best()

tr_loss, tr_ppl, tr_preds, \
tr_refs, tr_topics, tr_qs  = trainer.evaluate(train_loader)

vl_loss, vl_ppl, vl_preds, \
vl_refs, vl_topics, vl_qs  = trainer.evaluate(val_loader)

ts_loss, ts_ppl, ts_preds, \
ts_refs, ts_topics, ts_qs  = trainer.evaluate(test_loader)

print_metrics("Train",      tr_loss, tr_ppl,
              tr_preds, tr_refs, tr_topics, tr_qs)
print_metrics("Validation", vl_loss, vl_ppl,
              vl_preds, vl_refs, vl_topics, vl_qs)
print_metrics("Test",       ts_loss, ts_ppl,
              ts_preds, ts_refs, ts_topics, ts_qs)

# ── Sample Predictions from Test Set ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  Sample Predictions — Test Set (5 random)")
print(f"{'='*60}")

for i in np.random.choice(len(ts_preds), 5, replace=False):
    r = compute_rouge([ts_preds[i]], [ts_refs[i]])
    f = compute_token_f1(ts_preds[i], ts_refs[i])
    print(f"\n  Topic     : {ts_topics[i]}")
    print(f"  Question  : {ts_qs[i]}")
    print(f"  Reference : {ts_refs[i][:150]}...")
    print(f"  Generated : {ts_preds[i][:150]}...")
    print(f"  ROUGE-L   : {r['rougeL']:.4f} | Token F1 : {f:.4f}")
    print(f"  {'-'*55}")

# ══════════════════════════════════════════════════════════════════════════════
#  STANDALONE TEST QUERIES — one per topic
# ══════════════════════════════════════════════════════════════════════════════

def answer_pharma_query(question: str, k: int = 3) -> dict:
    retrieved   = retrieve_top_k(question, k=k)
    context_str = ' '.join([
        f"Context {i+1}: {r['answer']}"
        for i, r in enumerate(retrieved)
    ])
    input_text  = (
        f"Question: {question} "
        f"{context_str} "
        f"Compose a precise and accurate hospital pharmacy answer:"
    )

    bart_model.eval()
    inputs = bart_tokenizer(
        input_text,
        max_length     = 1024,
        truncation     = True,
        return_tensors = 'pt'
    ).to(device)

    with torch.no_grad():
        output = bart_model.generate(
            input_ids            = inputs['input_ids'],
            attention_mask       = inputs['attention_mask'],
            max_new_tokens       = 256,
            num_beams            = 4,
            early_stopping       = True,
            no_repeat_ngram_size = 3,
            length_penalty       = 1.0
        )

    answer = bart_tokenizer.decode(output[0], skip_special_tokens=True)
    return {
        'question'  : question,
        'answer'    : answer,
        'retrieved' : [{
            'topic'      : r['topic'],
            'question'   : r['question'],
            'similarity' : r['similarity']
        } for r in retrieved]
    }

test_queries = [
    # prescription
    "How do I collect my prescribed medicine from the AIIMS pharmacy?",
    # medicine_costs
    "Are medicines free for BPL patients at AIIMS?",
    # otc_medicines
    "Can I buy paracetamol without a prescription at the hospital pharmacy?",
    # side_effects
    "What should I do if I experience side effects from my medication?",
    # general_queries
    "What are the opening hours of the AIIMS pharmacy?"
]

print(f"\n{'='*60}")
print(f"  Standalone Test Queries — Pharmacy Model")
print(f"{'='*60}")

for q in test_queries:
    out = answer_pharma_query(q)
    print(f"\n  Q         : {out['question']}")
    print(f"  A         : {out['answer']}")
    print(f"  Retrieved :")
    for r in out['retrieved']:
        print(f"    [{r['similarity']}] "
              f"{r['topic']:25s} — {r['question'][:50]}...")
    print(f"  {'-'*55}")

Device : cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder        : sentence-transformers/all-MiniLM-L6-v2
Encoder params : 22,713,216  (frozen)

Building retrieval index...
Indexed 400 Q&A pairs across 5 topics
  general_queries           : 106 rows
  side_effects              : 78 rows
  otc_medicines             : 76 rows
  prescription              : 75 rows
  medicine_costs            : 65 rows


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


Composer       : GanjinZero/biobart-v2-base
Composer params: 166,404,864

Building RAG datasets...
Train : 280 samples | 70 batches
Val   : 60   samples | 15   batches
Test  : 60  samples | 15  batches

  Training Pharmacy BioBART RAG  |  15 epochs  |  Warmup 25  |  Total 255 steps
  Epoch | Train Loss | Train PPL | Val Loss | Val PPL | ROUGE-L | Status
  ----------------------------------------------------------------------
      1 |     0.7357 |    2.0869 |   0.0290 |  1.0295 |  0.9902 | ← best ✅
      2 |     0.0439 |    1.0449 |   0.0095 |  1.0096 |  0.9951 | ← best ✅
      3 |     0.0134 |    1.0135 |   0.0048 |  1.0048 |  0.9962 | ← best ✅
      4 |     0.0088 |    1.0089 |   0.0040 |  1.0040 |  0.9959 | ← best ✅
      5 |     0.0027 |    1.0027 |   0.0029 |  1.0029 |  0.9965 | ← best ✅
      6 |     0.0028 |    1.0028 |   0.0029 |  1.0029 |  0.9965 | patience 1/3
      7 |     0.0027 |    1.0027 |   0.0028 |  1.0028 |  0.9962 | ← best ✅
      8 |     0.0007 |    1.0007 |   0.00

In [ ]:
# ── Save Pharmacy Model Files ─────────────────────────────────────────────────
import os
import numpy as np
from google.colab import files

os.makedirs('saved_models/pharmacy', exist_ok=True)

# ── Save 1: Trained BioBART Weights ───────────────────────────────────────────
torch.save(
    bart_model.state_dict(),
    'saved_models/pharmacy/pharma_biobart_weights.pt'
)
print("✅ Saved: pharma_biobart_weights.pt")

# ── Save 2: Retrieval Embeddings ───────────────────────────────────────────────
np.save(
    'saved_models/pharmacy/pharma_qa_embeddings.npy',
    full_qa_embs
)
print("✅ Saved: pharma_qa_embeddings.npy")

# ── Save 3: Q&A Data ───────────────────────────────────────────────────────────
full_df.to_csv(
    'saved_models/pharmacy/pharma_qa_data.csv',
    index=False
)
print("✅ Saved: pharma_qa_data.csv")

# ── Verify Everything Before Downloading ──────────────────────────────────────
print("\n── Verification ──────────────────────────────────────────────────────")

# Check 1: weights file size
weights_size = os.path.getsize(
    'saved_models/pharmacy/pharma_biobart_weights.pt'
)
print(f"   pharma_biobart_weights.pt  : {weights_size / 1e6:.1f} MB")

# Check 2: embeddings shape
test_emb_load = np.load(
    'saved_models/pharmacy/pharma_qa_embeddings.npy'
)
print(f"   pharma_qa_embeddings.npy   : shape {test_emb_load.shape}")

# Check 3: csv rows
test_csv_load = pd.read_csv(
    'saved_models/pharmacy/pharma_qa_data.csv'
)
print(f"   pharma_qa_data.csv         : {len(test_csv_load)} rows, "
      f"columns: {list(test_csv_load.columns)}")

# Check 4: embeddings rows must match csv rows
assert test_emb_load.shape[0] == len(test_csv_load), \
    f"❌ MISMATCH — embeddings {test_emb_load.shape[0]} rows " \
    f"vs csv {len(test_csv_load)} rows"
print(f"\n✅ Verification passed — embeddings and csv rows match perfectly")

# ── Quick End-to-End Inference Test Before Downloading ────────────────────────
print("\n── Quick Inference Test ──────────────────────────────────────────────")

import torch.nn.functional as F
from transformers import BartForConditionalGeneration, BartTokenizer

# Reload weights into fresh model to simulate PyCharm loading
test_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
test_model.load_state_dict(
    torch.load(
        'saved_models/pharmacy/pharma_biobart_weights.pt',
        map_location=device
    )
)
test_model.to(device)
test_model.eval()

# Run one test query through the full pipeline
test_query     = "Can I get my prescription medicines free of cost at AIIMS?"
test_query_emb = encode_texts([test_query])

# Retrieve top 3
q_norm    = F.normalize(
    torch.tensor(test_query_emb, dtype=torch.float32), dim=-1
)
k_norm    = F.normalize(
    torch.tensor(test_emb_load,  dtype=torch.float32), dim=-1
)
sims      = torch.matmul(q_norm, k_norm.T).squeeze(0)
top_k_idx = torch.topk(sims, k=3).indices.tolist()

context_str = ' '.join([
    f"Context {i+1}: {test_csv_load.iloc[idx]['answer']}"
    for i, idx in enumerate(top_k_idx)
])

input_text = (
    f"Question: {test_query} "
    f"{context_str} "
    f"Compose a precise and accurate hospital pharmacy answer:"
)

inputs = bart_tokenizer(
    input_text,
    max_length     = 1024,
    truncation     = True,
    return_tensors = 'pt'
).to(device)

with torch.no_grad():
    output = test_model.generate(
        input_ids            = inputs['input_ids'],
        attention_mask       = inputs['attention_mask'],
        max_new_tokens       = 256,
        num_beams            = 4,
        early_stopping       = True,
        no_repeat_ngram_size = 3,
        length_penalty       = 1.0
    )

test_answer = bart_tokenizer.decode(output[0], skip_special_tokens=True)
print(f"   Query  : {test_query}")
print(f"   Answer : {test_answer[:200]}...")
print(f"\n✅ Inference test passed — model loads and generates correctly")

# ── Download All Three Files ───────────────────────────────────────────────────
print("\n── Downloading Files ─────────────────────────────────────────────────")
files.download('saved_models/pharmacy/pharma_biobart_weights.pt')
files.download('saved_models/pharmacy/pharma_qa_embeddings.npy')
files.download('saved_models/pharmacy/pharma_qa_data.csv')

print("\n✅ All 3 Pharmacy files downloaded.")
print("   → pharma_biobart_weights.pt")
print("   → pharma_qa_embeddings.npy")
print("   → pharma_qa_data.csv")

✅ Saved: pharma_biobart_weights.pt
✅ Saved: pharma_qa_embeddings.npy
✅ Saved: pharma_qa_data.csv

── Verification ──────────────────────────────────────────────────────
   pharma_biobart_weights.pt  : 666.1 MB
   pharma_qa_embeddings.npy   : shape (400, 384)
   pharma_qa_data.csv         : 400 rows, columns: ['question', 'answer', 'topic']

✅ Verification passed — embeddings and csv rows match perfectly

── Quick Inference Test ──────────────────────────────────────────────


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

   Query  : Can I get my prescription medicines free of cost at AIIMS?
   Answer : Inpatient medicines for admitted patients at AIIMS are generally provided as part of care at no or nominal cost; the ward pharmacist will advise on any charges for specific items....

✅ Inference test passed — model loads and generates correctly

── Downloading Files ─────────────────────────────────────────────────


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All 3 Pharmacy files downloaded.
   → pharma_biobart_weights.pt
   → pharma_qa_embeddings.npy
   → pharma_qa_data.csv


In [ ]:
# ── Save and Download All BioBART Weight Files ────────────────────────────────
import os
import torch
from google.colab import files, drive

# ── Mount Drive for permanent backup ──────────────────────────────────────────
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/hospital_chatbot_models'

# ── Helper function ────────────────────────────────────────────────────────────
def save_and_download_weights(model, local_path, drive_path, filename):
    # Save locally first
    torch.save(model.state_dict(), local_path)
    # Backup to Drive
    torch.save(model.state_dict(), drive_path)
    size = os.path.getsize(local_path) / 1e6
    print(f"✅ Saved: {filename}  ({size:.1f} MB)")
    print(f"   Drive : {drive_path}")
    print(f"   Local : {local_path}")

# ══════════════════════════════════════════════════════════════════════════════
#  IMPORTANT — READ THIS BEFORE RUNNING
# ══════════════════════════════════════════════════════════════════════════════
# This cell assumes all 5 models are still in memory from training.
# The variable names must match exactly what your training cells used:
#
#   Admin model     → bart_model (from admin training cell)
#   Billing model   → bart_model (from billing training cell)
#   DA model        → bart_model (from DA training cell)
#   Emergency model → bart_model (from emergency training cell)
#   Pharmacy model  → bart_model (from pharmacy training cell)
#
# Since all training cells use the same variable name bart_model,
# only the LAST trained model will be in memory as bart_model.
# So you need to save immediately after each training cell.
# This cell is for if you trained them all and saved checkpoints.
# ══════════════════════════════════════════════════════════════════════════════

os.makedirs('saved_weights', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/admin',              exist_ok=True)
os.makedirs(f'{SAVE_DIR}/billing',            exist_ok=True)
os.makedirs(f'{SAVE_DIR}/doctor_appointment', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/emergency',          exist_ok=True)
os.makedirs(f'{SAVE_DIR}/pharmacy',           exist_ok=True)

print("══════════════════════════════════════════════════════")
print("  Loading and saving weights from checkpoints")
print("══════════════════════════════════════════════════════\n")

# ── Load each from its checkpoint and save as clean weights file ──────────────
from transformers import BartForConditionalGeneration

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Admin ──────────────────────────────────────────────────────────────────────
print("Processing Admin...")
admin_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
ckpt = torch.load('best_admin_rag.pt', map_location=device)
admin_model.load_state_dict(ckpt['model_state'])
save_and_download_weights(
    admin_model,
    local_path = 'saved_weights/admin_biobart_weights.pt',
    drive_path = f'{SAVE_DIR}/admin/admin_biobart_weights.pt',
    filename   = 'admin_biobart_weights.pt'
)
del admin_model  # free memory
torch.cuda.empty_cache()
print()

# ── Billing ────────────────────────────────────────────────────────────────────
print("Processing Billing...")
billing_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
ckpt = torch.load('best_billing_rag.pt', map_location=device)
billing_model.load_state_dict(ckpt['model_state'])
save_and_download_weights(
    billing_model,
    local_path = 'saved_weights/billing_biobart_weights.pt',
    drive_path = f'{SAVE_DIR}/billing/billing_biobart_weights.pt',
    filename   = 'billing_biobart_weights.pt'
)
del billing_model
torch.cuda.empty_cache()
print()

# ── Doctor Appointment ─────────────────────────────────────────────────────────
print("Processing Doctor Appointment...")
da_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
ckpt = torch.load('best_da_rag.pt', map_location=device)
da_model.load_state_dict(ckpt['model_state'])
save_and_download_weights(
    da_model,
    local_path = 'saved_weights/da_biobart_weights.pt',
    drive_path = f'{SAVE_DIR}/doctor_appointment/da_biobart_weights.pt',
    filename   = 'da_biobart_weights.pt'
)
del da_model
torch.cuda.empty_cache()
print()

# ── Emergency ──────────────────────────────────────────────────────────────────
print("Processing Emergency...")
emergency_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
ckpt = torch.load('best_emergency_rag.pt', map_location=device)
emergency_model.load_state_dict(ckpt['model_state'])
save_and_download_weights(
    emergency_model,
    local_path = 'saved_weights/emergency_biobart_weights.pt',
    drive_path = f'{SAVE_DIR}/emergency/emergency_biobart_weights.pt',
    filename   = 'emergency_biobart_weights.pt'
)
del emergency_model
torch.cuda.empty_cache()
print()

# ── Pharmacy ───────────────────────────────────────────────────────────────────
print("Processing Pharmacy...")
pharma_model = BartForConditionalGeneration.from_pretrained(
    'GanjinZero/biobart-v2-base'
)
ckpt = torch.load('best_pharma_rag.pt', map_location=device)
pharma_model.load_state_dict(ckpt['model_state'])
save_and_download_weights(
    pharma_model,
    local_path = 'saved_weights/pharma_biobart_weights.pt',
    drive_path = f'{SAVE_DIR}/pharmacy/pharma_biobart_weights.pt',
    filename   = 'pharma_biobart_weights.pt'
)
del pharma_model
torch.cuda.empty_cache()
print()

# ══════════════════════════════════════════════════════════════════════════════
#  ALL SAVED TO DRIVE — Now download to machine
# ══════════════════════════════════════════════════════════════════════════════
print("══════════════════════════════════════════════════════")
print("  All weights saved to Drive successfully")
print("  Starting downloads to your machine...")
print("══════════════════════════════════════════════════════\n")
print("⚠️  Note: Each file is ~650MB. Downloads will be slow.")
print("   You can also download directly from Google Drive.\n")

files.download('saved_weights/admin_biobart_weights.pt')
print("✅ Downloading: admin_biobart_weights.pt")

files.download('saved_weights/billing_biobart_weights.pt')
print("✅ Downloading: billing_biobart_weights.pt")

files.download('saved_weights/da_biobart_weights.pt')
print("✅ Downloading: da_biobart_weights.pt")

files.download('saved_weights/emergency_biobart_weights.pt')
print("✅ Downloading: emergency_biobart_weights.pt")

files.download('saved_weights/pharma_biobart_weights.pt')
print("✅ Downloading: pharma_biobart_weights.pt")

print("\n✅ All 5 weight files processed.")
print("   If browser downloads are slow, get them from:")
print(f"   Google Drive → {SAVE_DIR}")




Mounted at /content/drive
══════════════════════════════════════════════════════
  Loading and saving weights from checkpoints
══════════════════════════════════════════════════════

Processing Admin...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

✅ Saved: admin_biobart_weights.pt  (666.1 MB)
   Drive : /content/drive/MyDrive/hospital_chatbot_models/admin/admin_biobart_weights.pt
   Local : saved_weights/admin_biobart_weights.pt

Processing Billing...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

✅ Saved: billing_biobart_weights.pt  (666.1 MB)
   Drive : /content/drive/MyDrive/hospital_chatbot_models/billing/billing_biobart_weights.pt
   Local : saved_weights/billing_biobart_weights.pt

Processing Doctor Appointment...


...

<IPython.core.display.Javascript object>

✅ Downloading: emergency_biobart_weights.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloading: pharma_biobart_weights.pt

✅ All 5 weight files processed.
   If browser downloads are slow, get them from:
   Google Drive → /content/drive/MyDrive/hospital_chatbot_models
